# 2-E（課後選做）：把整張分工圖在 Colab 跑起來，再看它怎麼上雲端

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AluminumShark/langgraph-skills-workshop/blob/main/notebooks/2-E_deploy.ipynb)

> 📍 **對應簡報**：第 2 堂第四站投影片 **37**「作業小助教的完整分工圖」、投影片 **48**「分工圖搬上雲端：每一格住在哪裡」，以及「補充：上雲端」投影片 **47–55**（課後自學）

## 🎯 這兩堂課的小專案：作業小助教

**作業小助教**是一個協助檢查 Python 作業的小程式。你交給它一段作業，它回覆檢查結果。
在流程版（1-A，workflow）中，語法有錯會先回覆錯誤；語法通過才請 AI 產生含**「優點：」與「改進：」**的回饋。
格式檢查若一直不通過，流程會在達到生成次數上限時停止，並標示未通過，不能把「停止」當成「合格」。

**先分清三件事：**語法正確（Python 看得懂）≠ 執行成功（跑的時候沒出錯）≠ 答案正確（符合題目要求）。

選它當練習，是因為它剛好三種東西都用得到（同一件事也可以組合起來做）：

| 小助教需要的 | 例子 | 我們用什麼做 |
| --- | --- | --- |
| 由流程控制的步驟 | 先檢查語法；有錯就回報，通過才產生回饋並檢查格式 | 用程式定義**流程**（LangGraph，第 1 堂） |
| 需要判斷的事 | 回饋怎麼寫、要不要真的執行程式看看 | 交給 **AI**（agent，第 1 堂） |
| 做法和規矩 | 回饋的語氣要溫和還是嚴格、好的變數名稱長什麼樣 | 寫成給 AI 看的**手冊**（Skill，第 2 堂）；能用程式檢查的規矩（例如變數命名）也可以附上腳本或寫進流程，2-B 會比較 |

👉 **這本 notebook 做的是：**把第 2 堂分工圖的簡化版（MCP、流程、批改 agent、驗證 agent、人工確認）跑一次：本版沒有題目附件與 read_spec，再看上雲端時每一塊住在哪裡。

<details><summary>想查零件名稱時再展開（不用先背）</summary>

| 零件 | 白話說它在做什麼 |
| --- | --- |
| `check_syntax`（檢查語法） | 只看程式的「文法」對不對，例如括號有沒有關好、冒號有沒有漏；**不會執行程式** |
| `explain_error`（解釋錯誤） | 語法錯的時候，用一句話告訴學生錯在第幾行 |
| `give_feedback`（寫回饋） | 在 1-A 接上模型後，請 AI 寫出「優點＋改進」回饋；是否符合格式由下一站檢查 |
| `check_format`（檢查格式） | 檢查兩個標記的順序、內容是否非空與總字數；不驗證回饋內容是否正確 |
| 工具：`run_code`、`check_indent`、`read_spec`（1-B 起）；`check_naming`（2-B 起） | 在教學限制下執行作業、檢查縮排、讀題目要求、檢查變數名稱 |
| 手冊：`friendly-feedback`、`strict-review`；`naming-check`（2-B）、`detailed-feedback`（2-C 資安實驗） | 告訴 AI 回饋要溫和鼓勵還是嚴格審查；命名檢查手冊附腳本；最後一本藏了可疑指令，用來練習檢查外部手冊 |

</details>

**操作提醒：**先執行「0. 環境設定」，再照本段的準備格 → 預測 → 執行 → 對照操作。
`%%writefile` 表示「把這格內容存成檔案」；修改後要執行才會生效。重跑準備格可能覆寫手冊，先存好自己的修改。

## 這本 notebook 在做什麼

第 2 堂最後，我們假設作業小助教要升級：從 GitHub 抓 200 份作業、一次批完、批完自動寄信。
我們把它拆成一張**分工圖**，從「由誰決定執行」和「能力如何提供」兩個角度分類；同一項功能可以同時用到幾種零件：

| 角色 | 白話說是什麼 | 這本裡的例子 |
| --- | --- | --- |
| 程式（流程） | 由流程控制的順序和檢查 | 一定先檢查語法；寄信前一定停下來等人確認 |
| 工具（tool） | AI 可以呼叫的 Python 函式 | `check_naming`（檢查命名）、`run_code`（執行作業） |
| MCP | 把「住在別的程式或服務裡的工具」接進來的標準插座 | 作業倉庫：抓作業、寄信 |
| 手冊（Skill） | 給 AI 看的工作手冊 | `friendly-feedback`（溫和鼓勵） |
| 批改 agent 與驗證 agent | 批改 agent 是同一套設定，每份作業用自己的輸入各跑一次（批次，不是多開角色），各有一張乾淨的「桌子」（自己的對話紀錄）；驗證 agent 是職責獨立、只檢查不撰寫的第二個角色 | 批改 agent（＝投影片 37 的回饋 agent）用 `friendly-feedback`；驗證 agent 在程式裡叫「稽核員」 |

課堂上只畫了圖，這本把它**真的跑一次**，而且全部在免費 Colab 裡：

| 練習 | 你會做什麼 | 對應投影片 |
| --- | --- | --- |
| 練習 1 | 開一個**真的 MCP server**「作業倉庫」：先用本機插座接，再把它變成一個網址（遠端 MCP） | 48、52 |
| 練習 2 | ⭐ **先批一份，再整批執行**：MCP 抓作業 → Send 分派各份 → 每份先查語法，通過才進批改與驗證 agent → 寄信前停下來等人確認 → MCP 寄信 | 31、34–37、48、53–54 |
| 練習 3 | 把它包成 **API 服務**：用網址開始批改、用網址確認寄信，thread_id 跨請求保留 | 48、49、51、54 |
| 練習 4 | **上線前檢查**，並產生部署檔（`Dockerfile`、`langgraph.json`） | 49–50、55 |

**這本不會真的部署到雲端**（那需要雲端帳號和信用卡）。MCP server、API 服務都開在 Colab 這台機器裡，
信也是假的，只會寫進 `/content/review-bot/寄件匣.txt`。

## 部署地圖：上雲端之後，每個東西住在哪裡

| 東西 | 在這本 notebook 裡 | 上雲端之後住在哪裡 | 練習 |
| --- | --- | --- | --- |
| 外層流程、批改 agent、驗證 agent、本地 tool（`check_naming`、`run_code`） | Colab 裡的 Python | **你的容器**（Cloud Run 或 LangSmith Deployment）：自建容器用 `server.py` 的 API（`POST /batch`）；代管平台改用平台內建的 threads／runs API | 2、3 |
| `skills/`、`scripts/` | `/content/review-bot/` | **跟著容器一起出貨**：改手冊 ＝ 發新版本 | 4 |
| 作業倉庫、寄信（MCP） | 同一台機器上的另一個程式 | **別人的服務**：遠端 MCP 透過網址連線，依服務設定授權 | 1 |
| 每個 thread_id 的 State、等人確認的信 | SQLite 檔 | **資料庫**（Postgres），機器換掉也還在 | 2、3 |
| 模型 API key | Colab 🔑 密鑰（或不需要） | **保管箱**（Secret Manager），變成環境變數 | 4 |
| 同時批幾份、Trace／Eval | `max_concurrency`、印出來的用量 | **任務佇列**、**監控儀表板** | 2、4 |

一句話：**程式、手冊、腳本跟著容器走；會變的資料進資料庫；鑰匙進保管箱；別人的能力透過網路接進來。**

表裡的名詞：**容器**＝把程式和它需要的一切裝進同一個箱子，搬到哪台機器都能跑；**Cloud Run、LangSmith Deployment**＝幫你在雲端跑程式的服務；
**Postgres**＝一種資料庫；**Secret Manager**＝雲端的密碼保管箱；**任務佇列**＝工作太多時讓它們排隊；
**Trace／Eval**＝記錄每一步做了什麼／自動考試打分數。

**讀結果時注意：**「驗證通過」表示回饋檢查通過，不代表學生作業正確。語法錯誤的作業由程式直接產生錯誤通知，也可列為可寄出的回饋。
2-E 固定使用預設溫和手冊，批改後進行回饋驗證；2-D 才示範選手冊與完整格式重做。

## 怎麼用

1. 先執行 **「0. 環境設定」**（會多裝 MCP、FastAPI；並準備好前面幾本的工具和手冊）。
2. 建議先做練習 1 的 stdio，再做練習 2 的一份作業與人工確認；能說明結果與確認步驟後才選整批。HTTP、API、部署檔留第二輪。
3. 模型呼叫數隨份數、工具使用與退回重寫改變；先用一份觀察用量。環境設定使用全新預設手冊，不會延續前面 notebook 的修改。
4. 重跑環境設定與 `%%writefile` 準備格會覆寫其中的預設檔案，請先保留修改；SQLite 存檔與假的寄件匣不會因此清空。環境重建後要重新準備檔案與服務。
5. 這本用到 `await`：Colab 的格子可以直接寫 `await`，不用管它，照樣按 ▶ 就好。

In [ ]:
#@title 0. 環境設定（第一次打開、或執行階段重設後執行）
#@markdown **模型來源**：`colab` ＝ Colab 內建模型（不另填 API key；可用性與額度依帳號）；`gemini` ＝ 講師提供的 Gemini API key（備案）
模型來源 = "colab"  #@param ["colab", "gemini"]
#@markdown 選 `gemini` 時：建議在左邊 🔑「密鑰」新增 `GOOGLE_API_KEY`；也可以把 key 貼在下面（⚠️ 貼了就不要把 notebook 分享出去）
講師_API_KEY = ""  #@param {type:"string"}
#@markdown 進階（通常不用改）：指定模型名稱，留空用預設
colab_模型名稱 = ""  #@param {type:"string"}
gemini_模型名稱 = "gemini-flash-latest"  #@param {type:"string"}

import os, shutil, subprocess, sys
from pathlib import Path

# 1. 用 uv 安裝套件（限制相容版本範圍；時間依網路而異）
if not os.environ.get("WORKSHOP_SKIP_INSTALL"):
    print("⏳ 用 uv 安裝 LangGraph / LangChain ...")
    if shutil.which("uv") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(f'uv pip install --python "{sys.executable}" -q "langgraph>=1.1,<2" "langchain>=1.2,<2" "langchain-google-genai>=4,<5" "langgraph-checkpoint-sqlite>=3,<4" "fastapi>=0.115" "uvicorn>=0.30" "mcp>=1.20,<2" "langchain-mcp-adapters>=0.3,<0.4"', shell=True, check=True)

# 2. 寫出這本 notebook 用到的共用檔案
RB = Path("/content/review-bot")
for d in ['tools', 'deploy', 'skills/friendly-feedback', 'skills/strict-review', 'skills/naming-check/scripts', 'skills/detailed-feedback/scripts']:
    (RB / d).mkdir(parents=True, exist_ok=True)
_檔案 = {
    'llm.py': r'''"""模型轉接頭：把「Colab 內建模型」或「講師的 Gemini API key」包成同一個 model 物件。

學生不用看這個檔案。後面所有 cell 都只寫 model.invoke(...)、model.bind_tools(...)，
不管背後是哪一家模型，用法都一樣。

- colab ：google.colab.ai.generate_text（免費帳號可用，不需要 API key）
- gemini：langchain_google_genai.ChatGoogleGenerativeAI（需要 GOOGLE_API_KEY）

generate_text 只吃一段文字、只回一段文字，不支援原生的 tool calling。
所以 ColabAIChat 做了一個小轉接：
1. 把多則訊息接成一段文字（系統指示、使用者、AI、工具結果各標一個標題）
2. 有綁工具時，在文字最後附上工具清單，請模型「需要工具就只回一行 JSON」
3. 把模型回的 JSON 解析成 tool_calls，格式和原生 tool calling 一模一樣
這剛好讓你看到：模型從頭到尾只是在「說」它想用哪個工具。
"""

from __future__ import annotations

import json
import os
import re
import time
import uuid
from pathlib import Path
from typing import Any

from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.utils.function_calling import convert_to_openai_tool

設定檔 = Path(__file__).with_name(".llm.json")
預設_gemini_模型 = "gemini-flash-latest"


def 估算_token(text: str) -> int:
    """粗估 token 數：中日韓文字約 1 字 1 token，其他字元約 4 個 1 token。"""
    cjk = len(re.findall(r"[⺀-鿿豈-﫿＀-￯]", text))
    return max(1, round(cjk + (len(text) - cjk) / 4))


# ---------- 把訊息接成一段文字 ----------

def _工具說明(tools: list[dict]) -> str:
    lines = []
    for t in tools:
        fn = t.get("function", t)
        props = fn.get("parameters", {}).get("properties", {})
        params = ", ".join(f"{k}: {v.get('type', 'string')}" for k, v in props.items())
        desc = " ".join((fn.get("description") or "").split())
        lines.append(f"- {fn['name']}({params})：{desc}")
    return "\n".join(lines)


def _訊息轉文字(messages: list[BaseMessage], tools: list[dict] | None, tool_choice: Any) -> str:
    parts = []
    for m in messages:
        if m.type == "system":
            parts.append(f"【系統指示】\n{m.text}")
        elif m.type == "human":
            parts.append(f"【使用者】\n{m.text}")
        elif m.type == "ai":
            if m.tool_calls:
                calls = [{"name": c["name"], "args": c["args"]} for c in m.tool_calls]
                parts.append("【AI】\n" + json.dumps({"tool_calls": calls}, ensure_ascii=False))
            else:
                parts.append(f"【AI】\n{m.text}")
        elif m.type == "tool":
            name = getattr(m, "name", None) or "工具"
            parts.append(f"【工具 {name} 的結果】\n{m.text}")
    if tools:
        must = ""
        if tool_choice in ("any", "required"):
            must = "這一次你一定要呼叫至少一個工具。\n"
        elif isinstance(tool_choice, str) and tool_choice not in ("auto", "none"):
            must = f"這一次你一定要呼叫 {tool_choice}。\n"
        parts.append(
            "【你可以用的工具】\n"
            f"{_工具說明(tools)}\n\n"
            "【回覆規則】\n"
            f"{must}"
            "如果你需要使用工具，只回一行 JSON，不要加任何其他文字、也不要加 ``` 符號：\n"
            '{"tool_calls": [{"name": "工具名稱", "args": {"參數名稱": "參數值"}}]}\n'
            "如果不需要工具（例如已經有足夠的資訊），就直接用一般文字回答，不要輸出 JSON。"
        )
    parts.append("【AI】")
    return "\n\n".join(parts)


def _解析回覆(text: str, tools: list[dict] | None) -> AIMessage:
    """模型回的是 JSON 紙條就轉成 tool_calls，否則當成一般文字。"""
    meta = {"raw_text": text, "model_provider": "colab"}
    if tools:
        s = text.strip()
        s = re.sub(r"^```(?:json)?\s*|\s*```$", "", s)
        start, end = s.find("{"), s.rfind("}")
        if start != -1 and end > start and ("tool_calls" in s or s.startswith("{")):
            try:
                data = json.loads(s[start : end + 1])
            except json.JSONDecodeError:
                data = None
            calls = None
            if isinstance(data, dict):
                if isinstance(data.get("tool_calls"), list):
                    calls = data["tool_calls"]
                elif "name" in data or "tool" in data:
                    calls = [data]
            if calls:
                tool_calls = []
                for c in calls:
                    if not isinstance(c, dict):
                        continue
                    name = c.get("name") or c.get("tool")
                    args = c.get("args") or c.get("arguments") or {}
                    if isinstance(args, str):
                        try:
                            args = json.loads(args)
                        except json.JSONDecodeError:
                            args = {}
                    if name:
                        tool_calls.append(
                            {"name": name, "args": args, "id": f"call_{uuid.uuid4().hex[:8]}", "type": "tool_call"}
                        )
                if tool_calls:
                    return AIMessage(content="", tool_calls=tool_calls, response_metadata=meta)
    return AIMessage(content=text, response_metadata=meta)


class ColabAIChat(BaseChatModel):
    """把 google.colab.ai.generate_text 包成 LangChain 的聊天模型。"""

    model_name: str | None = None
    max_retries: int = 3

    @property
    def _llm_type(self) -> str:
        return "colab-ai"

    def bind_tools(self, tools, *, tool_choice=None, **kwargs):
        formatted = [convert_to_openai_tool(t) for t in tools]
        return self.bind(tools=formatted, tool_choice=tool_choice)

    def _generate(self, messages, stop=None, run_manager=None, **kwargs) -> ChatResult:
        from google.colab import ai

        tools = kwargs.get("tools")
        prompt = _訊息轉文字(messages, tools, kwargs.get("tool_choice"))
        extra = {"model_name": self.model_name} if self.model_name else {}
        for attempt in range(self.max_retries):
            try:
                text = ai.generate_text(prompt, **extra) or ""
                break
            except Exception:
                if attempt == self.max_retries - 1:
                    raise
                time.sleep(2 * (attempt + 1))
        message = _解析回覆(text, tools)
        message.usage_metadata = {
            "input_tokens": 估算_token(prompt),
            "output_tokens": 估算_token(text),
            "total_tokens": 估算_token(prompt) + 估算_token(text),
        }
        message.response_metadata["token_count"] = "估算值（Colab 內建模型不回報用量）"
        return ChatResult(generations=[ChatGeneration(message=message)])


# ---------- 對外：設定與取得 model ----------

def 讀設定() -> dict:
    try:
        return json.loads(設定檔.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        return {}


def 存設定(provider: str, colab_model: str = "", gemini_model: str = "") -> None:
    """記住選了哪個模型來源（不存 API key）。"""
    設定檔.write_text(
        json.dumps(
            {"provider": provider, "colab_model": colab_model, "gemini_model": gemini_model or 預設_gemini_模型},
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


def _找_api_key() -> str | None:
    if os.environ.get("GOOGLE_API_KEY"):
        return os.environ["GOOGLE_API_KEY"]
    try:
        from google.colab import userdata

        key = userdata.get("GOOGLE_API_KEY")
        if key:
            os.environ["GOOGLE_API_KEY"] = key
            return key
    except Exception:
        pass
    return None


def get_model() -> BaseChatModel:
    cfg = 讀設定()
    # 上雲端時沒有設定檔：用環境變數 WORKSHOP_PROVIDER=gemini（金鑰從 GOOGLE_API_KEY 讀）
    provider = os.environ.get("WORKSHOP_PROVIDER") or cfg.get("provider", "colab")
    if provider == "gemini":
        key = _找_api_key()
        if not key:
            raise RuntimeError(
                "選了 gemini 但找不到 API key：請在左邊 🔑「密鑰」新增 GOOGLE_API_KEY，"
                "或在「0. 環境設定」的表單貼上講師提供的 key，再重跑那一格。"
            )
        from langchain_google_genai import ChatGoogleGenerativeAI

        return ChatGoogleGenerativeAI(
            model=cfg.get("gemini_model") or 預設_gemini_模型, google_api_key=key, max_retries=3
        )
    try:
        from google.colab import ai  # noqa: F401
    except ImportError as exc:
        raise RuntimeError("google.colab.ai 只能在 Google Colab 上使用；不在 Colab 的話請改選 gemini。") from exc
    return ColabAIChat(model_name=cfg.get("colab_model") or None)
''',
    'workshop.py': r'''"""工作坊 notebook 共用的小工具（學生不用看）。

- 準備()          ：每一段的第一格都會呼叫，重新連上模型、重新載入改過的檔案
- 範例作業         ：表單下拉選單裡的幾段作業
- 記下() / 印出歷程()：保存實驗格指定的結果，最後一格按紀錄標題列出
- 顯示流程圖()      ：把 LangGraph 畫成圖
- 存到雲端硬碟() / 下載壓縮檔()
"""

from __future__ import annotations

import datetime as _dt
import importlib
import json
import shutil
import sys
from pathlib import Path

from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import BaseMessage

from llm import get_model, 估算_token  # noqa: F401  （讓 notebook 可以直接 from workshop import 估算_token）

__all__ = [
    "BASE", "準備", "範例作業", "取得作業", "記下", "印出狀態", "印出訂單紙", "印出歷程", "清除歷程",
    "模型呼叫計數器", "顯示流程圖", "存到雲端硬碟", "下載壓縮檔", "估算_token",
]

BASE = Path(__file__).resolve().parent
歷程檔 = BASE / "history.jsonl"

範例作業 = {
    "少了右括號": "print('hi'\n",
    "正確的程式": "def add(a, b):\n    return a + b\n\nprint(add(2, 3))\n",
    "空白": "",
    "變數打錯字": "def add(a, b):\n    total = a + b\n    return summ\n\nprint(add(2, 3))\n",
    "縮排亂掉": "def average(scores):\n  total = 0\n  for s in scores:\n        total += s\n  return total / len(scores)\n\nprint(average([90, 85, 77]))\n",
    "命名不一致": (
        "def CalcAverage(scoreList):\n"
        "    Total = 0\n"
        "    for s in scoreList:\n"
        "        Total += s\n"
        "    avgScore = Total / len(scoreList)\n"
        "    return avgScore\n"
        "\n"
        "myScores = [90, 85, 77]\n"
        "MAX = 100\n"
        "result_2 = CalcAverage(myScores)\n"
        "print(result_2, '/', MAX)\n"
    ),
}


def 取得作業(名稱: str) -> str:
    """表單選「我的作業」時讀 /content/review-bot/我的作業.py，其他從範例裡拿。"""
    if 名稱.startswith("我的作業"):
        path = BASE / "我的作業.py"
        if not path.exists():
            raise FileNotFoundError("還沒有「我的作業」：請先執行上面寫著 %%writefile ... 我的作業.py 的那一格。")
        return path.read_text(encoding="utf-8")
    return 範例作業[名稱]


def 準備():
    """每一段的起點：把 review-bot 放進 import 路徑、重新載入改過的檔案、回傳 model。"""
    if str(BASE) not in sys.path:
        sys.path.insert(0, str(BASE))
    importlib.invalidate_caches()
    for name, mod in list(sys.modules.items()):
        path = getattr(mod, "__file__", None) or ""
        if path and Path(path).resolve().is_relative_to(BASE) and name not in ("workshop", "llm"):
            try:
                importlib.reload(mod)
            except Exception:
                sys.modules.pop(name, None)
    _讓_mcp_stdio_能在notebook用()
    return get_model()


def _讓_mcp_stdio_能在notebook用() -> None:
    """mcp 的 stdio_client 會把 MCP server 的錯誤輸出接到 sys.stderr。
    Colab（Linux）的 sys.stderr 有真的檔案代號，沒問題；Windows 的 Jupyter 沒有，就改接到 log 檔。"""
    try:
        sys.stderr.fileno()
        return
    except Exception:
        pass
    try:
        import mcp.client.stdio as mcp_stdio
    except ImportError:
        return
    (BASE / "deploy").mkdir(exist_ok=True)
    log = open(BASE / "deploy" / "mcp_server.log", "a", encoding="utf-8")
    mcp_stdio.stdio_client.__wrapped__.__defaults__ = (log,)


# ---------- State 歷程 ----------

def _可存的(obj):
    if isinstance(obj, BaseMessage):
        d = {"type": obj.type, "content": obj.text}
        if getattr(obj, "tool_calls", None):
            d["tool_calls"] = [{"name": c["name"], "args": c["args"]} for c in obj.tool_calls]
        if obj.type == "tool":
            d["name"] = getattr(obj, "name", None)
        return d
    if isinstance(obj, dict):
        return {str(k): _可存的(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_可存的(v) for v in obj]
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    return str(obj)


def 記下(標題: str, 內容) -> None:
    """把這一次的結果（通常是 State）存進 history.jsonl。"""
    row = {"時間": _dt.datetime.now().strftime("%H:%M:%S"), "標題": 標題, "內容": _可存的(內容)}
    with 歷程檔.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


def _短(text, n=300):
    text = str(text)
    return text if len(text) <= n else text[:n] + f"…（還有 {len(text) - n} 字）"


def 印出狀態(state: dict, 縮排: str = "  ") -> None:
    for k, v in state.items():
        if k == "messages":
            print(f"{縮排}{k}: （{len(v)} 則訊息）")
            for m in v:
                m = _可存的(m)
                if isinstance(m, dict) and m.get("tool_calls"):
                    print(f"{縮排}  [{m['type']}] 呼叫 {m['tool_calls']}")
                elif isinstance(m, dict):
                    print(f"{縮排}  [{m.get('type')}] {_短(m.get('content', ''), 200)}")
        else:
            print(f"{縮排}{k}: {_短(repr(v) if not isinstance(v, str) else repr(v), 300)}")


def 印出訂單紙(第幾步: int, state: dict, 上一步: dict | None = None, 不印: tuple = ("code",)) -> None:
    """stream_mode="values" 每一步都會給整張訂單紙；順便標出這一步新寫了哪幾格。"""
    if not 上一步:
        說明 = "起點：空白訂單紙"
    else:
        新寫入 = [k for k in state if state.get(k) != 上一步.get(k)]
        說明 = "新寫入：" + ("、".join(新寫入) if 新寫入 else "（沒有改變）")
    print(f"第 {第幾步} 步（{說明}）")
    print("  ", {k: (_短(v, 120) if isinstance(v, str) else v) for k, v in state.items() if k not in 不印})


def 印出歷程() -> None:
    if not 歷程檔.exists():
        print("還沒有任何紀錄：先跑上面幾段，再回來執行這一格。")
        return
    rows = [json.loads(line) for line in 歷程檔.read_text(encoding="utf-8").splitlines() if line.strip()]
    print(f"📸 共 {len(rows)} 筆紀錄（存在 {歷程檔}）\n")
    for i, row in enumerate(rows, 1):
        print(f"===== {i}. [{row['時間']}] {row['標題']} =====")
        內容 = row["內容"]
        if isinstance(內容, dict):
            印出狀態(內容)
        elif isinstance(內容, list):
            for j, item in enumerate(內容, 1):
                print(f"  --- 第 {j} 步 ---")
                if isinstance(item, dict):
                    印出狀態(item, "    ")
                else:
                    print(f"    {_短(item)}")
        else:
            print(f"  {_短(內容, 1000)}")
        print()


def 清除歷程() -> None:
    歷程檔.unlink(missing_ok=True)


# ---------- 數模型被呼叫了幾次 ----------

class 模型呼叫計數器(BaseCallbackHandler):
    """放進 config={"callbacks": [計數器]}，每呼叫一次模型就 +1。"""

    def __init__(self):
        self.次數 = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        self.次數 += 1

    def on_llm_start(self, serialized, prompts, **kwargs):
        self.次數 += 1


# ---------- 畫流程圖 ----------

def 顯示流程圖(graph) -> None:
    g = graph.get_graph()
    try:
        from IPython.display import Image, display

        display(Image(g.draw_mermaid_png()))
    except Exception:
        print("（畫圖服務暫時連不上，改用文字版流程圖；可以貼到 https://mermaid.live 看圖）\n")
        print(g.draw_mermaid())


# ---------- 保存 ----------

def 存到雲端硬碟() -> None:
    from google.colab import drive

    drive.mount("/content/drive")
    目的地 = Path("/content/drive/MyDrive/langgraph-skills-workshop") / _dt.datetime.now().strftime("%Y%m%d-%H%M%S")
    shutil.copytree(BASE, 目的地, ignore=shutil.ignore_patterns("__pycache__", ".llm.json"))
    print(f"✅ 已存到雲端硬碟：{目的地}")
    print("   （notebook 本身請用上方選單「檔案 → 在雲端硬碟中儲存副本」）")


def 下載壓縮檔() -> None:
    from google.colab import files

    名稱 = BASE.parent / ("review-bot-" + _dt.datetime.now().strftime("%Y%m%d-%H%M%S"))
    zip_path = shutil.make_archive(str(名稱), "zip", BASE.parent, BASE.name)
    print(f"✅ 已打包：{zip_path}")
    files.download(zip_path)
''',
    'tools/__init__.py': r'''"""作業小助教的工具箱：run_code、check_indent、read_spec、check_naming。"""
''',
    'tools/run_code.py': r'''"""run_code：在教學限制下執行學生的作業。

學生作業裡什麼都可能有（import os 刪檔、無限迴圈），所以這個工具有兩道執行限制，加上一個方便閱讀的設計：
1. 執行前先檢查：只准 import 少數標準模組（math、random、re…），os / sys / subprocess 這類一律擋下；
   也不准用 eval、exec、open、input 和底線開頭的屬性（常被拿來繞過限制）
2. 開一個獨立的子程序執行，超過時間就強制停止（無限迴圈也停得下來）
3. 只回一行結果（第幾行、什麼錯），方便人與模型閱讀；這不是安全限制

⚠️ 這是教學用的圍欄。正式產品請用真正的 sandbox（容器、RestrictedPython、e2b、Modal）。
"""

import ast
import json
import os
import subprocess
import sys
import tempfile

from langchain.tools import tool

禁止的模組 = {"os", "sys", "subprocess", "shutil", "socket", "pathlib", "ctypes", "importlib",
           "multiprocessing", "threading", "signal", "pickle", "builtins", "io", "inspect", "gc"}
允許的模組 = {"math", "random", "string", "re", "datetime", "collections", "itertools", "functools",
           "statistics", "time", "decimal", "fractions", "json", "copy", "heapq", "bisect"}
禁止的函式 = {"eval", "exec", "compile", "open", "input", "__import__", "globals", "locals", "vars",
           "getattr", "setattr", "delattr", "breakpoint", "help", "exit", "quit", "memoryview"}
禁止的屬性 = {"system", "popen", "modules", "builtins", "f_globals", "f_back", "gi_frame", "cr_frame"}
最長時間_秒 = 3
最長輸出_字 = 800

# 子程序裡真正執行作業的那段程式：用「縮小版的 builtins」執行，import 只放行白名單
_RUNNER = r"""
import builtins, contextlib, io, json, sys, traceback
code = sys.stdin.buffer.read().decode("utf-8")
allowed = set(json.loads(sys.argv[1]))
blocked = set(json.loads(sys.argv[2]))
real_import = builtins.__import__
def guarded_import(name, globals=None, locals=None, fromlist=(), level=0):
    if name.split(".")[0] not in allowed:
        raise ImportError("這個工具不允許 import " + name)
    return real_import(name, globals, locals, fromlist, level)
safe = {k: v for k, v in vars(builtins).items() if k not in blocked}
safe["__import__"] = guarded_import
try:
    import resource
    resource.setrlimit(resource.RLIMIT_AS, (512 * 1024 * 1024, 512 * 1024 * 1024))
except Exception:
    pass
out = io.StringIO()
result = {"ok": True}
try:
    compiled = compile(code, "作業.py", "exec")
    with contextlib.redirect_stdout(out):
        exec(compiled, {"__builtins__": safe, "__name__": "__main__"})
except SyntaxError as e:
    result = {"ok": False, "line": e.lineno, "error": "SyntaxError: " + str(e.msg)}
except BaseException as e:
    line = None
    for frame in traceback.extract_tb(e.__traceback__):
        if frame.filename == "作業.py":
            line = frame.lineno
    result = {"ok": False, "line": line, "error": type(e).__name__ + ": " + str(e)}
result["stdout"] = out.getvalue()
sys.__stdout__.write(json.dumps(result))
"""


def _先檢查(code: str) -> str | None:
    """執行前的靜態檢查；有問題就回一行說明，沒問題回 None。"""
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return f"❌ 第 {e.lineno} 行 SyntaxError: {e.msg}"
    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            names = [a.name for a in node.names] if isinstance(node, ast.Import) else [node.module or ""]
            for name in names:
                root = name.split(".")[0]
                if root in 禁止的模組 or root not in 允許的模組:
                    return f"⛔ 已擋下：第 {node.lineno} 行 import {name}（這個工具不允許使用 {root}）"
        elif isinstance(node, ast.Name) and node.id in 禁止的函式:
            return f"⛔ 已擋下：第 {node.lineno} 行用了 {node.id}（這個工具不允許）"
        elif isinstance(node, ast.Attribute) and (
            node.attr.startswith("_") or node.attr in 禁止的屬性 or node.attr in 禁止的模組
        ):
            return f"⛔ 已擋下：第 {node.lineno} 行存取了 .{node.attr}（這個工具不允許）"
    return None


def 執行作業(code: str, 最長時間: float = 最長時間_秒) -> str:
    """真正做事的函式：檢查 → 子程序執行 → 回傳一行結果。"""
    if not code.strip():
        return "❌ 作業是空白的，沒有東西可以執行"
    擋下 = _先檢查(code)
    if 擋下:
        return 擋下
    env = {"PYTHONIOENCODING": "utf-8", "PATH": os.environ.get("PATH", "")}
    if os.name == "nt":
        env["SYSTEMROOT"] = os.environ.get("SYSTEMROOT", "")
    try:
        proc = subprocess.run(
            [sys.executable, "-I", "-c", _RUNNER, json.dumps(sorted(允許的模組)), json.dumps(sorted(禁止的函式))],
            input=code, capture_output=True, text=True, encoding="utf-8",
            timeout=最長時間, cwd=tempfile.gettempdir(), env=env,
        )
    except subprocess.TimeoutExpired:
        return f"⏱️ 執行超過 {最長時間:g} 秒被強制停止（可能有無限迴圈）"
    try:
        result = json.loads(proc.stdout.strip().splitlines()[-1])
    except (IndexError, json.JSONDecodeError):
        return "❌ 執行失敗（程式沒有正常結束，可能用了太多記憶體）"
    stdout = result.get("stdout", "")
    if len(stdout) > 最長輸出_字:
        stdout = stdout[:最長輸出_字] + "\n…（輸出太長，後面省略）"
    if result["ok"]:
        return f"✅ 執行成功，輸出：\n{stdout}" if stdout else "✅ 執行成功（沒有印出任何東西）"
    where = f"第 {result['line']} 行 " if result.get("line") else ""
    印出 = f"\n（出錯前印出：{stdout.strip()}）" if stdout.strip() else ""
    return f"❌ {where}{result['error']}{印出}"


@tool
def run_code(code: str) -> str:
    """在教學限制下執行一段 Python 作業，回傳輸出或錯誤訊息。

    什麼時候用：想知道作業「實際跑起來」會印出什麼、會不會出錯時。
    參數 code：完整的 Python 程式碼（字串）。
    回傳：一行結果，例如「✅ 執行成功，輸出：5」或「❌ 第 3 行 NameError: ...」。
    限制：只能 import math、random、re 等少數標準模組；不能用 input()、open()；超過 3 秒會被停止。
    """
    return 執行作業(code)
''',
    'tools/check_naming.py': r'''"""check_naming：把 Skill 的腳本 scripts/check_naming.py 包成 agent 的工具。

腳本外面包一層 @tool，就進了 agent 的工具箱：
本例由 AI 決定是否呼叫工具並整理結果；Python 腳本在相同版本與輸入下回傳相同檢查結果，
但仍要確認模型是否實際呼叫、是否完整保留結果。
"""

import subprocess
import sys
from pathlib import Path

from langchain.tools import tool

腳本 = Path(__file__).resolve().parent.parent / "skills" / "naming-check" / "scripts" / "check_naming.py"


def 執行命名腳本(code: str) -> str:
    proc = subprocess.run(
        [sys.executable, str(腳本)], input=code.encode("utf-8"), capture_output=True, timeout=10
    )
    out = proc.stdout.decode("utf-8").strip()
    return out or "腳本執行失敗：" + proc.stderr.decode("utf-8", "replace").strip()[-200:]


@tool
def check_naming(code: str) -> str:
    """檢查變數與函式名稱是否為小寫加底線（snake_case），回傳不合規則的行。

    什麼時候用：批改作業、需要列出命名問題時（這是 naming-check 手冊附的腳本）。
    參數 code：完整的 Python 程式碼（字串）。
    回傳：每個問題一行，例如「第 3 行：myVar 不符合規則（建議 my_var）」；全部合格時回傳「OK：…」。
    """
    return 執行命名腳本(code)
''',
    'graph.py': r'''"""1-A：作業小助教的 workflow，路線由程式決定。

START → check_syntax ─語法 OK→ give_feedback → check_format ─格式 OK 或達上限→ END
                     └語法錯→ explain_error → END     └不合格、retries 還沒到上限→ 退回 give_feedback

三個零件：
- State：一張訂單紙（dict），從頭傳到尾
- Node ：一個工作站（函式），收 State、只回傳自己改的那幾格
- Edge ：工作站之間的箭頭；條件邊就是一個回傳「下一站名字」的 if
"""

import ast
import re
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class State(TypedDict, total=False):
    code: str              # 學生交的作業
    syntax_ok: bool | None # 語法對不對（還沒檢查時是 None）
    error: str             # 語法錯在哪裡
    feedback: str          # 給學生的回饋
    format_ok: bool | None # 回饋格式合不合格
    format_problem: str    # 不合格的原因（退回重寫時會告訴 AI）
    retries: int           # 已產生回饋的次數，包含第一次


# ---------- Node 1：check_syntax 檢查語法（純 Python）：只看文法對不對，不執行程式 ----------

_錯誤翻譯 = [
    (r"'(.)' was never closed", lambda m: f"少了右括號 {dict(zip('([{', ')]}')).get(m.group(1), '')}"),
    (r"unterminated string literal", lambda m: "字串少了結尾的引號"),
    (r"expected ':'", lambda m: "少了冒號 :"),
    (r"unexpected indent", lambda m: "縮排多了"),
    (r"expected an indented block", lambda m: "這裡需要縮排"),
    (r"unindent does not match", lambda m: "縮排對不齊"),
    (r"unmatched '(.)'", lambda m: f"多了一個 {m.group(1)}"),
    (r"invalid character", lambda m: "有全形或看不懂的字元"),
    (r"Perhaps you forgot a comma", lambda m: "可能少了逗號"),
    (r"invalid syntax", lambda m: "語法不正確"),
]


def 翻譯語法錯誤(msg: str) -> str:
    for pattern, 說法 in _錯誤翻譯:
        m = re.search(pattern, msg)
        if m:
            return 說法(m)
    return msg


def check_syntax(state: State) -> dict:
    code = state.get("code", "")
    if not code.strip():
        return {"syntax_ok": False, "error": "作業是空白的，沒有任何程式"}
    try:
        ast.parse(code)
    except SyntaxError as e:
        return {"syntax_ok": False, "error": f"第 {e.lineno} 行{翻譯語法錯誤(e.msg)}"}
    return {"syntax_ok": True, "error": ""}


# ---------- 條件邊 1：語法對 → 寫回饋；語法錯 → 解釋錯誤（就是一個 if） ----------

def route_after_syntax(state: State) -> str:
    if state["syntax_ok"]:
        return "give_feedback"
    return "explain_error"


# ---------- Node 2：explain_error 解釋錯誤（純 Python）：一句話說錯在第幾行 ----------

def explain_error(state: State) -> dict:
    return {"feedback": f"語法：{state['error']}\n改進：先把這一行修好，再執行一次看看。"}


# ---------- Node 3：give_feedback 寫回饋（兩個版本：固定文字／交給 AI） ----------

def give_feedback_固定文字(state: State) -> dict:
    """先不接 AI 的版本：回固定的一段話。"""
    return {
        "feedback": "優點：這份作業通過語法檢查。\n改進：（這一版還沒接 AI，尚未檢查執行結果與邏輯）",
        "retries": state.get("retries", 0) + 1,
    }


def 建立_give_feedback(model, 回饋字數上限: int):
    """接上 AI 的版本：AI 只是住在這個節點裡的一行程式。"""

    def give_feedback(state: State) -> dict:
        prompt = (
            "你是 Python 作業小助教，請用繁體中文給下面這段作業回饋。\n"
            "格式規定（一定要遵守）：\n"
            "- 只寫兩段：第一段以「優點：」開頭，第二段以「改進：」開頭\n"
            f"- 整段回饋（含「優點：」「改進：」與換行）不超過 {回饋字數上限} 個字元\n\n"
            f"作業：\n```python\n{state['code']}\n```"
        )
        if state.get("format_problem"):
            prompt += f"\n\n你上一版的回饋被退回，原因：{state['format_problem']}。請修正後重寫。"
        text = model.invoke(prompt).text
        return {"feedback": text.strip(), "retries": state.get("retries", 0) + 1}

    return give_feedback


# ---------- Node 4：check_format 檢查格式（純 Python 的檢查站）：要有兩段、不能太長 ----------

def 建立_check_format(回饋字數上限: int):
    def check_format(state: State) -> dict:
        feedback = state.get("feedback", "")
        問題 = []
        # 檢查標籤順序與兩段非空內容；接受全形／半形冒號與換行。
        parts = re.fullmatch(r"\s*優點[：:]\s*(.+?)\s*改進[：:]\s*(.+?)\s*", feedback, re.DOTALL)
        if not parts or not all(part.strip() for part in parts.groups()):
            問題.append("需依序寫出「優點：內容」與「改進：內容」兩段")
        if len(feedback) > 回饋字數上限:
            問題.append(f"太長了：{len(feedback)} 字元（含標籤與換行），上限 {回饋字數上限} 字元")
        return {"format_ok": not 問題, "format_problem": "；".join(問題)}

    return check_format


# ---------- 條件邊 2：格式 OK 或回饋產生次數到上限 → 結束；否則退回重寫 ----------

def 建立_route_after_format(最多重試次數: int):
    def route_after_format(state: State) -> str:
        if state["format_ok"]:
            return END
        if state.get("retries", 0) >= 最多重試次數:
            return END  # 已達回饋產生上限；停止不代表格式通過
        return "give_feedback"

    return route_after_format


# ---------- 把零件接起來 ----------

def 建立小助教(model=None, 最多重試次數: int = 3, 回饋字數上限: int = 150):
    """model=None 時用固定文字。最多重試次數是歷史參數名，實際包含第一次產生。"""
    give_feedback = 建立_give_feedback(model, 回饋字數上限) if model else give_feedback_固定文字

    builder = StateGraph(State)
    builder.add_node("check_syntax", check_syntax)
    builder.add_node("explain_error", explain_error)
    builder.add_node("give_feedback", give_feedback)
    builder.add_node("check_format", 建立_check_format(回饋字數上限))

    builder.add_edge(START, "check_syntax")
    builder.add_conditional_edges("check_syntax", route_after_syntax, ["give_feedback", "explain_error"])
    builder.add_edge("explain_error", END)
    builder.add_edge("give_feedback", "check_format")
    builder.add_conditional_edges("check_format", 建立_route_after_format(最多重試次數), ["give_feedback", END])
    return builder.compile()


def 空白訂單紙(code: str) -> State:
    return {"code": code, "syntax_ok": None, "error": "", "feedback": "", "format_ok": None,
            "format_problem": "", "retries": 0}
''',
    'skill_loader.py': r'''"""skill_loader：AI 選手冊只看「名片」，選中了才翻開內文。

漸進式揭露三層：
1. 名片：SKILL.md 開頭兩條 --- 之間的 name 和 description，挑選時提供給模型
2. 內文：整份 SKILL.md，被選中才讀
3. 附錄：scripts/、references/ 裡的檔案，用到才打開
"""

from pathlib import Path

SKILLS = Path(__file__).resolve().parent / "skills"


def 讀名片(path: Path) -> dict:
    """只讀兩條 --- 之間的 name 和 description。"""
    text = Path(path).read_text(encoding="utf-8")
    card = {"name": Path(path).parent.name, "description": "", "path": str(path)}
    if text.startswith("---"):
        head = text.split("---", 2)[1]
        for line in head.strip().splitlines():
            key, _, value = line.partition(":")
            if key.strip() in ("name", "description"):
                card[key.strip()] = value.strip()
    return card


def 讀內文(card: dict) -> str:
    """選中之後才翻開：名片以下的手冊內容。"""
    text = Path(card["path"]).read_text(encoding="utf-8")
    return text.split("---", 2)[2].strip() if text.startswith("---") else text


def 列出名片(只要: list[str] | None = None) -> list[dict]:
    cards = [讀名片(p) for p in sorted(SKILLS.glob("*/SKILL.md"))]
    if 只要:
        cards = [c for c in cards if Path(c["path"]).parent.name in 只要]
    return cards


def 挑手冊(model, 需求: str, cards: list[dict]) -> dict:
    """把所有名片給模型看，讓它挑一本；回傳被挑中的那張名片。"""
    if not cards:
        raise ValueError("找不到可用手冊；請先寫出 SKILL.md，再列出名片。")
    清單 = "\n".join(f"- {c['name']}：{c['description']}" for c in cards)
    回答 = model.invoke(
        "你要替下面的批改需求挑一本最適合的手冊。\n"
        f"批改需求：{需求}\n\n"
        f"手冊名片：\n{清單}\n\n"
        "只回答手冊的 name，不要其他字。"
    ).text.strip()
    名稱 = 回答.strip("`\"' \n")
    選項 = {c["name"]: c for c in cards}
    if 名稱 not in 選項:
        raise ValueError(f"模型沒有回傳唯一可用的手冊名稱：{回答!r}。可用名稱：{list(選項)}。請檢查名片與模型輸出；本次未選取手冊。")
    return 選項[名稱]
''',
    'spec.md': r'''請寫一個函式 add(a, b)，回傳兩個數字相加的結果，並用 print(add(2, 3)) 印出 5。
''',
    'deploy/__init__.py': r'''"""2-E：上雲端前的準備（API 服務、平行批改、寄信前人工確認、上線前檢查）。"""
''',
    'deploy/homework_mcp.py': r'''"""作業倉庫 MCP server：工具住在「另一個程式」裡，透過 MCP 插座接進來。

它就是一個普通的 Python 檔，只是函式上面加了 @mcp.tool()。
真實世界裡，這個角色是 GitHub 的 MCP server（抓作業）和信箱的 MCP server（寄信）；
這裡用假資料代替，寄信也只會寫進 寄件匣.txt。

兩種接法：
    python homework_mcp.py              # stdio：本機插座，由 agent 那邊把它開起來
    python homework_mcp.py --http 8765  # Streamable HTTP：變成一個網址，像遠端 MCP
"""

import argparse
import datetime as _dt
import json
from pathlib import Path

from mcp.server.fastmcp import FastMCP

寄件匣 = Path(__file__).resolve().parent.parent / "寄件匣.txt"

# 假的作業倉庫：學生、信箱、交的作業
作業們 = {
    "小明": ("xiaoming@example.com", "print('hi'\n"),
    "小華": ("xiaohua@example.com", "def add(a, b):\n    return a + b\n\nprint(add(2, 3))\n"),
    "小美": ("xiaomei@example.com", "def add(a, b):\n    total = a + b\n    return summ\n\nprint(add(2, 3))\n"),
    "阿傑": ("ajie@example.com", "def CalcAverage(scoreList):\n    Total = 0\n    for s in scoreList:\n        Total += s\n"
                                 "    return Total / len(scoreList)\n\nmyScores = [90, 85, 77]\nprint(CalcAverage(myScores))\n"),
    "小芳": ("xiaofang@example.com", "def average(scores):\n  total = 0\n  for s in scores:\n        total += s\n"
                                     "  return total / len(scores)\n\nprint(average([90, 85, 77]))\n"),
}

mcp = FastMCP("作業倉庫", log_level="WARNING")


@mcp.tool()
def list_homework() -> str:
    """列出已經交作業的學生（名字和 email），回傳 JSON 字串。要批改作業時先呼叫這個。"""
    return json.dumps([{"學生": name, "email": email} for name, (email, _) in 作業們.items()], ensure_ascii=False)


@mcp.tool()
def get_homework(student: str) -> str:
    """取得某位學生交的作業程式碼。參數 student：學生名字（從 list_homework 拿）。"""
    if student not in 作業們:
        return f"找不到「{student}」的作業"
    return 作業們[student][1]


@mcp.tool()
def send_email(to: str, subject: str, body: str) -> str:
    """寄信給學生（不可逆：寄出去就收不回來）。參數：收件人 email、主旨、內文。"""
    with 寄件匣.open("a", encoding="utf-8") as f:
        f.write(f"[{_dt.datetime.now():%H:%M:%S}] 寄給 {to}｜{subject}\n{body}\n\n")
    return f"（模擬）已寄給 {to}"


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--http", type=int, help="用 Streamable HTTP 開在這個 port（不給就用 stdio）")
    args = parser.parse_args()
    if args.http:
        mcp.settings.host = "127.0.0.1"
        mcp.settings.port = args.http
        mcp.run(transport="streamable-http")   # 網址：http://127.0.0.1:<port>/mcp
    else:
        mcp.run()
''',
    'deploy/multi_agent.py': r'''"""2-E 練習 2：把第 2 堂的整張分工圖在 Colab 跑一次（多 agent ＋ MCP）。

START → 抓作業 ─Send×N→ 批一份 → 寄信前確認 ─要寄→ 寄信 → END
         (MCP)             │          (interrupt)  └都不要寄→ END   (MCP)
                           └ 程式檢查語法 → 批改 agent（手冊＋腳本＋run_code）→ 驗證 agent

對照分工圖中可以組合的五種角色（本版是簡化版：沒有題目附件與 read_spec）：
- 程式：外層流程的順序、語法檢查、「驗證不過就重寫一次」、選「都不要寄」就直接結束
- Tool：check_naming（手冊附的腳本）、run_code
- MCP ：作業倉庫的 list_homework / get_homework / send_email（工具住在另一個程式裡）
- Skill：friendly-feedback 手冊，當批改 agent 的 system prompt
- agent：同一套批改 agent 逐份處理（Send 分派，每份一張乾淨的桌子）＋ 一個職責獨立的驗證 agent
"""

import json
import operator
import re
import sys
from pathlib import Path
from typing import Annotated, TypedDict

from langchain.agents import create_agent
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send, interrupt

from graph import check_syntax
from skill_loader import 列出名片, 讀內文
from tools.check_naming import check_naming
from tools.run_code import run_code

MCP_SERVER = Path(__file__).resolve().with_name("homework_mcp.py")
存檔位置 = Path(__file__).resolve().with_name("multi_agent.sqlite")
寄信選項 = ["全部寄出", "只寄驗證通過的", "都不要寄"]


# ---------- MCP：作業倉庫的插座 ----------

def 作業倉庫設定(網址: str | None = None) -> dict:
    """給網址就連遠端（Streamable HTTP），沒給就在本機開一個（stdio）。"""
    if 網址:
        return {"作業倉庫": {"url": 網址, "transport": "streamable_http"}}
    return {"作業倉庫": {"command": sys.executable, "args": [str(MCP_SERVER)], "transport": "stdio"}}


async def 取得MCP工具(設定: dict) -> dict:
    return {t.name: t for t in await MultiServerMCPClient(設定).get_tools()}


def 結果文字(結果) -> str:
    """MCP 工具回傳的是一串內容區塊，取出裡面的文字。"""
    if isinstance(結果, str):
        return 結果
    if isinstance(結果, list):
        return "".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in 結果)
    return str(結果)


# ---------- State ----------

class 總State(TypedDict, total=False):
    名單: list[dict]                             # [{"學生", "email", "code"}]
    結果們: Annotated[list[dict], operator.add]   # 每個「批一份」交回來的結果，往下接著寫
    決定: str
    寄件紀錄: list[str]


class 一份State(TypedDict):
    學生: str
    email: str
    code: str


# ---------- 驗證 agent：獨立檢查作業與回饋，不負責撰寫回饋 ----------

稽核員人設 = (
    "你是嚴格的稽核員，只負責檢查別人寫的作業回饋，自己不寫回饋。\n"
    "檢查三件事：1. 回饋說的問題在作業裡真的存在嗎 2. 有沒有亂講行號 3. 有沒有用「錯誤」「很差」這類字眼。\n"
    '只回一行 JSON，不要其他文字：{"通過": true 或 false, "原因": "一句話"}'
)


def 驗證(model, code: str, feedback: str) -> dict:
    # 程式能判斷的，先用程式檢查（這幾項不花 token）；稽核員每份都會再呼叫一次模型
    問題 = []
    if "優點" not in feedback or "改進" not in feedback:
        問題.append("少了「優點」或「改進」段落")
    行數 = len(code.splitlines())
    for n in re.findall(r"第\s*(\d+)\s*行", feedback):
        if not 1 <= int(n) <= 行數:
            問題.append(f"提到第 {n} 行，但作業只有 {行數} 行")
    # 程式判斷不了的，交給稽核員
    帶行號 = "\n".join(f"{i:>2} | {line}" for i, line in enumerate(code.splitlines(), 1))
    回答 = model.invoke([SystemMessage(稽核員人設), HumanMessage(f"作業：\n{帶行號}\n\n回饋：\n{feedback}")]).text
    m = re.search(r"\{.*\}", 回答, re.S)
    try:
        判斷 = json.loads(m.group(0)) if m else {}
    except json.JSONDecodeError:
        判斷 = {}
    if not isinstance(判斷, dict) or not isinstance(判斷.get("通過"), bool):
        問題.append("稽核結果無法解析成布林通過值，請人工檢查")
    elif not 判斷["通過"]:
        問題.append("稽核員：" + str(判斷.get("原因", 回答.strip()[:60])))
    return {"通過": not 問題, "原因": "；".join(問題) or "OK"}


# ---------- 把零件接起來 ----------

def 建立多agent小助教(model, mcp設定: dict | None = None, checkpointer=None, 只批學生: str | None = None):
    mcp設定 = mcp設定 or 作業倉庫設定()
    手冊 = 讀內文(列出名片(["friendly-feedback"])[0])
    批改員人設 = (
        f"{手冊}\n\n"
        "# 你的工具\n"
        "- check_naming：檢查變數命名（手冊附的腳本），要談命名時用它\n"
        "- run_code：執行作業看輸出\n"
        "要不要用、用哪一個，你自己決定。回饋不超過 250 字。"
    )

    async def 抓作業(state: 總State) -> dict:
        # 抓作業不需要判斷，所以是「程式」呼叫 MCP 工具，不是 agent
        tools = await 取得MCP工具(mcp設定)
        名單 = json.loads(結果文字(await tools["list_homework"].ainvoke({})))
        if 只批學生 is not None:
            名單 = [a for a in 名單 if a["學生"] == 只批學生]
            if not 名單:
                raise ValueError(f"作業倉庫沒有學生：{只批學生}")
        for a in 名單:
            a["code"] = 結果文字(await tools["get_homework"].ainvoke({"student": a["學生"]}))
        return {"名單": 名單}

    def 分派(state: 總State) -> list[Send]:
        # 每份作業開一個「批一份」：各自一張乾淨的桌子
        return [Send("批一份", a) for a in state["名單"]]

    def 批一份(state: 一份State) -> dict:
        code = state["code"]
        語法 = check_syntax({"code": code})
        if not 語法["syntax_ok"]:   # 能寫成 if 的事：程式直接回，不叫 AI
            feedback = f"語法：{語法['error']}\n改進：先把這一行修好，再執行一次看看。"
            return {"結果們": [{**state, "feedback": feedback, "用到的工具": [], "重寫次數": 0,
                              "驗證": {"通過": True, "原因": "語法錯誤由程式直接回覆，沒有經過 AI"}}]}
        批改員 = create_agent(model, tools=[check_naming, run_code], system_prompt=批改員人設)
        messages = [{"role": "user", "content": f"請批改這份作業：\n```python\n{code}\n```"}]
        用到的工具, 重寫次數 = [], 0
        while True:
            result = 批改員.invoke({"messages": messages}, config={"recursion_limit": 25})
            # result 含前一輪完整紀錄；只取本輪新增訊息，避免重複計入舊工具要求。
            用到的工具 += [c["name"] for m in result["messages"][len(messages):] for c in (getattr(m, "tool_calls", None) or [])]
            feedback = result["messages"][-1].text.strip()
            結論 = 驗證(model, code, feedback)
            if 結論["通過"] or 重寫次數 >= 1:      # 驗證不過最多退回重寫一次
                break
            重寫次數 += 1
            messages = result["messages"] + [{"role": "user", "content": f"稽核員退回：{結論['原因']}。請修正後重寫回饋。"}]
        return {"結果們": [{**state, "feedback": feedback, "用到的工具": 用到的工具,
                          "重寫次數": 重寫次數, "驗證": 結論}]}

    def 寄信前確認(state: 總State) -> dict:
        # 寄信不可逆：停下來，把要寄的內容交給人看
        清單 = [{"學生": r["學生"], "email": r["email"], "驗證通過": r["驗證"]["通過"],
                "回饋": r["feedback"]} for r in state["結果們"]]
        決定 = interrupt({"要寄出的信": 清單, "選項": 寄信選項})
        if 決定 == "都不要寄":
            return {"決定": 決定, "寄件紀錄": ["沒有寄出任何信（決定：都不要寄）"]}
        return {"決定": 決定}

    def 要不要寄(state: 總State) -> str:
        # 選「都不要寄」就直接結束，不進寄信節點
        return END if state.get("決定") == "都不要寄" else "寄信"

    async def 寄信(state: 總State) -> dict:
        要寄 = [r for r in state["結果們"]
              if state.get("決定") == "全部寄出" or (state.get("決定") == "只寄驗證通過的" and r["驗證"]["通過"])]
        if not 要寄:
            return {"寄件紀錄": [f"沒有寄出任何信（決定：{state.get('決定')}）"]}
        tools = await 取得MCP工具(mcp設定)
        紀錄 = []
        for r in 要寄:
            結果 = await tools["send_email"].ainvoke({"to": r["email"], "subject": "作業回饋", "body": r["feedback"]})
            紀錄.append(結果文字(結果))
        return {"寄件紀錄": 紀錄}

    builder = StateGraph(總State)
    builder.add_node("抓作業", 抓作業)
    builder.add_node("批一份", 批一份)
    builder.add_node("寄信前確認", 寄信前確認)
    builder.add_node("寄信", 寄信)
    builder.add_edge(START, "抓作業")
    builder.add_conditional_edges("抓作業", 分派, ["批一份"])
    builder.add_edge("批一份", "寄信前確認")
    builder.add_conditional_edges("寄信前確認", 要不要寄, ["寄信", END])
    builder.add_edge("寄信", END)
    return builder.compile(checkpointer=checkpointer)


async def 開存檔(path: Path = 存檔位置):
    """interrupt 要等人確認，所以 State 一定要存起來；這裡存成 SQLite 檔（雲端改用 Postgres）。"""
    import aiosqlite
    from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

    return AsyncSqliteSaver(await aiosqlite.connect(str(path)))


class 用量計(BaseCallbackHandler):
    """數模型被呼叫幾次、加總 token（Colab 內建模型的 token 是估算值）。"""

    def __init__(self):
        self.次數 = 0
        self.input_tokens = 0
        self.output_tokens = 0

    def on_llm_end(self, response, **kwargs):
        self.次數 += 1
        for generations in response.generations:
            for g in generations:
                usage = getattr(getattr(g, "message", None), "usage_metadata", None) or {}
                self.input_tokens += usage.get("input_tokens", 0)
                self.output_tokens += usage.get("output_tokens", 0)
''',
    'skills/friendly-feedback/SKILL.md': r'''---
name: friendly-feedback
description: 給 Python 初學者的作業回饋。語氣溫和，先講優點再講改進。適合第一次交作業、剛學會 Python 的同學。
---

# 溫和回饋規則

1. 先講一個做得好的地方
2. 最多指出兩個問題，每個都附上改法
3. 不使用「錯誤」「很差」這類字眼
4. 回饋分兩段：第一段以「優點：」開頭，第二段以「改進：」開頭
''',
    'skills/strict-review/SKILL.md': r'''---
name: strict-review
description: 期中考、正式評分用的嚴格審查。列出所有問題、不留情面、附上扣分。
---

# 嚴格審查規則

1. 列出所有問題，一個都不放過，每個問題都標出第幾行
2. 不說客套話，不用「很棒」「不錯」這類字眼
3. 每個問題附上扣幾分（滿分 100 分）
4. 回饋分兩段：第一段以「優點：」開頭（真的沒有就寫「優點：無」），第二段以「改進：」開頭
''',
    'skills/naming-check/SKILL.md': r'''---
name: naming-check
description: 檢查 Python 作業的變數與函式名稱是不是小寫英文加底線（snake_case）。批改作業、需要列出命名問題時使用。
---

# 命名檢查

執行 scripts/check_naming.py（工具名稱 check_naming），把它的輸出寫進回饋。
不要自己判斷、不要增加或刪掉任何一項，照抄它的每一行。

共同規則：檢查賦值變數、函式名稱和函式參數；同名只列第一次定義，不檢查類別名稱，略過 self、cls。
名稱可有一個開頭底線，接小寫英文字母，後面可接小寫英文字母、數字或底線。
本練習不設常數例外：MAX 不合格；result_2 合格。這些規則由腳本執行。

輸出格式：照抄腳本的每一行，例如「第 3 行：myVar 不符合規則（建議 my_var）」
''',
    'skills/naming-check/scripts/check_naming.py': r'''"""檢查變數與函式名稱是不是小寫英文加底線（snake_case）。

用法：
    python check_naming.py 作業.py        # 檢查一個檔案
    python check_naming.py < 作業.py      # 或從標準輸入讀
輸出：
    每個問題一行，例如「第 3 行：myVar 不符合規則（建議 my_var）」
    全部合格時印「OK：所有變數與函式名稱都是小寫加底線」

一個腳本只做一件事、輸出短短幾行，方便人與 AI 核對；相同輸入與相同腳本的檢查結果相同。
"""

import ast
import re
import sys

規則 = re.compile(r"^_?[a-z][a-z0-9_]*$")


def 建議名稱(name: str) -> str:
    s = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", name)
    return re.sub(r"_+", "_", s).lower()


def 找出所有名稱(tree: ast.AST) -> dict[str, int]:
    """名稱 → 第一次出現的行號（變數、函式、參數，不含 class 名稱）。"""
    名稱 = {}

    def 記(name, line):
        if name not in 名稱 or line < 名稱[name]:
            名稱[name] = line

    for node in ast.walk(tree):
        if isinstance(node, ast.Name) and isinstance(node.ctx, ast.Store):
            記(node.id, node.lineno)
        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            記(node.name, node.lineno)
        elif isinstance(node, ast.arg) and node.arg not in ("self", "cls"):
            記(node.arg, node.lineno)
    return 名稱


def 檢查命名(code: str) -> str:
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return f"無法檢查：第 {e.lineno} 行有語法錯誤"
    問題 = [
        f"第 {line} 行：{name} 不符合規則（建議 {建議名稱(name)}）"
        for name, line in sorted(找出所有名稱(tree).items(), key=lambda x: (x[1], x[0]))
        if not 規則.match(name)
    ]
    return "\n".join(問題) if 問題 else "OK：所有變數與函式名稱都是小寫加底線"


if __name__ == "__main__":
    if len(sys.argv) > 1:
        with open(sys.argv[1], encoding="utf-8") as f:
            source = f.read()
    else:
        source = sys.stdin.buffer.read().decode("utf-8")
    sys.stdout.reconfigure(encoding="utf-8")
    print(檢查命名(source))
''',
}
for rel, content in _檔案.items():
    (RB / rel).write_text(content, encoding="utf-8")
sys.path.insert(0, str(RB))
# 剛寫好的檔案要讓 Python 重新找一次（如果之前先跑過別的格子，Python 會記得「這個資料夾不存在」）
import importlib
importlib.invalidate_caches()

# 3. 記住模型來源（不會存 API key），測試連線
import llm
llm.存設定(模型來源, colab_模型名稱.strip(), gemini_模型名稱.strip())
if 講師_API_KEY.strip():
    os.environ["GOOGLE_API_KEY"] = 講師_API_KEY.strip()
from workshop import *
model = 準備()

回覆 = model.invoke("用一句話跟剛學 Python 的同學打招呼").text
print(f"✅ 環境設定完成，模型來源：{模型來源}")
print(f"🤖 模型說：{回覆.strip()}")

---
## 練習 1：開一個真的 MCP server

> 📍 **對應簡報**：投影片 48（部署地圖）、52（遠端 MCP）；第 1 堂投影片 39（MCP 插座）

先把分工圖上「別人的服務」那一格做出來。MCP 的工具住在**另一個程式**裡。下一格寫出 `deploy/homework_mcp.py`：它就是一個普通的 Python 檔，
三個函式上面加了 `@mcp.tool()`，就變成可以從插座接出去的工具：

本練習的資料源是 `homework_mcp.py` 中的 Python 字典，預先放好 **5 位學生的假信箱與作業程式碼**，尚未讀取 GitHub 上的學生 repo。
MCP server 與連線是真的，資料來源與寄信則使用模擬內容。

| 工具 | 本練習實作 | 真正接服務時 |
| --- | --- | --- |
| `list_homework`、`get_homework` | 從預設字典取名單與程式碼 | 需接 GitHub API，或轉接相容的 MCP 工具 |
| `send_email` | 只寫進模擬寄件匣 | 需串接真正的寄信服務 |

這些是本課自訂的工具名稱；不能只把網址換成任意 GitHub MCP server 就直接相容，還要對應工具名稱、參數與回傳格式。

兩種接法：**stdio** 是本機插座（agent 這邊把 server 開成一個子程式）；**Streamable HTTP** 是一個網址，
遠端 MCP 就是這樣，只是實際服務可能使用 **OAuth** 登入或其他授權方式（用帳號登入、換一張通行證的標準做法）。

### 🤔 預測題
1. 接上之後，MCP 工具和你在 1-B 自己寫的 `@tool` 看起來有什麼不一樣？
2. 換成網址接，工具的名字和用法會變嗎？

In [2]:
%%writefile /content/review-bot/deploy/homework_mcp.py
"""作業倉庫 MCP server：工具住在「另一個程式」裡，透過 MCP 插座接進來。

它就是一個普通的 Python 檔，只是函式上面加了 @mcp.tool()。
真實世界裡，這個角色是 GitHub 的 MCP server（抓作業）和信箱的 MCP server（寄信）；
這裡用假資料代替，寄信也只會寫進 寄件匣.txt。

兩種接法：
    python homework_mcp.py              # stdio：本機插座，由 agent 那邊把它開起來
    python homework_mcp.py --http 8765  # Streamable HTTP：變成一個網址，像遠端 MCP
"""

import argparse
import datetime as _dt
import json
from pathlib import Path

from mcp.server.fastmcp import FastMCP

寄件匣 = Path(__file__).resolve().parent.parent / "寄件匣.txt"

# 假的作業倉庫：學生、信箱、交的作業
作業們 = {
    "小明": ("xiaoming@example.com", "print('hi'\n"),
    "小華": ("xiaohua@example.com", "def add(a, b):\n    return a + b\n\nprint(add(2, 3))\n"),
    "小美": ("xiaomei@example.com", "def add(a, b):\n    total = a + b\n    return summ\n\nprint(add(2, 3))\n"),
    "阿傑": ("ajie@example.com", "def CalcAverage(scoreList):\n    Total = 0\n    for s in scoreList:\n        Total += s\n"
                                 "    return Total / len(scoreList)\n\nmyScores = [90, 85, 77]\nprint(CalcAverage(myScores))\n"),
    "小芳": ("xiaofang@example.com", "def average(scores):\n  total = 0\n  for s in scores:\n        total += s\n"
                                     "  return total / len(scores)\n\nprint(average([90, 85, 77]))\n"),
}

mcp = FastMCP("作業倉庫", log_level="WARNING")


@mcp.tool()
def list_homework() -> str:
    """列出已經交作業的學生（名字和 email），回傳 JSON 字串。要批改作業時先呼叫這個。"""
    return json.dumps([{"學生": name, "email": email} for name, (email, _) in 作業們.items()], ensure_ascii=False)


@mcp.tool()
def get_homework(student: str) -> str:
    """取得某位學生交的作業程式碼。參數 student：學生名字（從 list_homework 拿）。"""
    if student not in 作業們:
        return f"找不到「{student}」的作業"
    return 作業們[student][1]


@mcp.tool()
def send_email(to: str, subject: str, body: str) -> str:
    """寄信給學生（不可逆：寄出去就收不回來）。參數：收件人 email、主旨、內文。"""
    with 寄件匣.open("a", encoding="utf-8") as f:
        f.write(f"[{_dt.datetime.now():%H:%M:%S}] 寄給 {to}｜{subject}\n{body}\n\n")
    return f"（模擬）已寄給 {to}"


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--http", type=int, help="用 Streamable HTTP 開在這個 port（不給就用 stdio）")
    args = parser.parse_args()
    if args.http:
        mcp.settings.host = "127.0.0.1"
        mcp.settings.port = args.http
        mcp.run(transport="streamable-http")   # 網址：http://127.0.0.1:<port>/mcp
    else:
        mcp.run()


Overwriting /content/review-bot/deploy/homework_mcp.py


In [3]:
#@title ▶ 執行：用本機插座（stdio）接上作業倉庫
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
import json
from langchain_core.utils.function_calling import convert_to_openai_tool
from deploy import multi_agent

設定 = multi_agent.作業倉庫設定()          # stdio：由這邊把 homework_mcp.py 開成另一個程式
工具們 = await multi_agent.取得MCP工具(設定)
for t in 工具們.values():
    print(f"🔌 {t.name}（參數：{list(t.args) or '無'}）：{t.description.splitlines()[0]}")

名單 = json.loads(multi_agent.結果文字(await 工具們["list_homework"].ainvoke({})))
print("\nlist_homework →", "、".join(a["學生"] for a in 名單))
print("\nget_homework(student='小美') →")
print(multi_agent.結果文字(await 工具們["get_homework"].ainvoke({"student": "小美"})))

說明書 = 估算_token(json.dumps([convert_to_openai_tool(t) for t in 工具們.values()], ensure_ascii=False))
print(f"這 {len(工具們)} 個 MCP 工具的說明書合計：約 {說明書} token（估算）")
記下("2-E 練習 1：stdio 接上作業倉庫", {"工具": list(工具們), "學生": [a["學生"] for a in 名單], "說明書token": 說明書})

🔌 list_homework（參數：無）：列出已經交作業的學生（名字和 email），回傳 JSON 字串。要批改作業時先呼叫這個。
🔌 get_homework（參數：['student']）：取得某位學生交的作業程式碼。參數 student：學生名字（從 list_homework 拿）。
🔌 send_email（參數：['to', 'subject', 'body']）：寄信給學生（不可逆：寄出去就收不回來）。參數：收件人 email、主旨、內文。

list_homework → 小明、小華、小美、阿傑、小芳

get_homework(student='小美') →
def add(a, b):
    total = a + b
    return summ

print(add(2, 3))

這 3 個 MCP 工具的說明書合計：約 244 token（估算）


In [4]:
#@title ▶ 選做：同一個 server 換成一個網址（Streamable HTTP，像遠端 MCP）
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
import asyncio, socket, subprocess
from deploy import multi_agent

舊的 = getattr(sys, "_作業倉庫程序", None)       # 重跑時先把上一次開的關掉
if 舊的:
    舊的.terminate()
with socket.socket() as s:
    s.bind(("127.0.0.1", 0))
    port = s.getsockname()[1]
紀錄檔 = open(BASE / "deploy" / "homework_mcp.log", "w")
sys._作業倉庫程序 = subprocess.Popen([sys.executable, str(multi_agent.MCP_SERVER), "--http", str(port)],
                                 stdout=紀錄檔, stderr=紀錄檔)
網址 = f"http://127.0.0.1:{port}/mcp"
for _ in range(60):
    try:
        工具們 = await multi_agent.取得MCP工具(multi_agent.作業倉庫設定(網址))
        break
    except Exception:
        await asyncio.sleep(0.5)
else:
    raise SystemExit("⚠️ MCP server 沒有開起來，可以看 /content/review-bot/deploy/homework_mcp.log")

(BASE / "deploy" / ".mcp_port").write_text(str(port))
print(f"🌐 作業倉庫 MCP 現在是一個網址：{網址}")
print("用網址接上的工具：", "、".join(工具們))
print("\nget_homework(student='阿傑') →")
print(multi_agent.結果文字(await 工具們["get_homework"].ainvoke({"student": "阿傑"})))
記下("2-E 練習 1：網址接上作業倉庫", {"網址": 網址, "工具": list(工具們)})

🌐 作業倉庫 MCP 現在是一個網址：http://127.0.0.1:48879/mcp
用網址接上的工具： list_homework、get_homework、send_email

get_homework(student='阿傑') →
def CalcAverage(scoreList):
    Total = 0
    for s in scoreList:
        Total += s
    return Total / len(scoreList)

myScores = [90, 85, 77]
print(CalcAverage(myScores))



### 👀 你看到了什麼

- MCP 工具一樣有名字、說明、參數，**接成工具後，模型同樣依名字、說明與參數提出呼叫要求**；執行時還涉及另一個程式的連線與錯誤處理。
  （這份 Python adapter 範例使用 `await ...ainvoke()` 等待工具回傳；這不是所有 MCP 客戶端的固定寫法。）
- 換成網址接，工具一模一樣。上雲端時，作業倉庫就是別人服務上的一個網址：你的程式不用知道它怎麼寫、用什麼語言寫。
- MCP 提供工具清單；由應用程式決定把哪些工具提供給模型。Skill 可以描述使用方式，但不會自動縮減工具定義或撤銷權限。這份流程由程式直接呼叫 MCP，批改 agent 只拿到命名檢查與執行工具。
- **完成檢查**：先在 stdio 輸出找到工具名稱與小美的作業。能說明「另一個程式提供工具」後，再選做網址接法；不必先懂網路部署。

---
## 練習 2 ⭐：把整張分工圖跑一次

> 📍 **對應簡報**：投影片 31「功能 1：從 GitHub 抓作業」、34「批 200 份：先獨立執行，再考慮並行」、35「驗證回饋：先規則，再必要的獨立判斷」、36「寄信前，讓流程等待確認」、37「作業小助教的完整分工圖」、48「分工圖搬上雲端：每一格住在哪裡」、53「200 份同時批：Send」、54「寄信前等待確認：interrupt」

練習 1 把作業倉庫開起來了，這一個練習用它把分工圖的簡化版跑一次。下一格寫出 `deploy/multi_agent.py`；本版沒有題目附件，也沒有 `read_spec`：

| 分工圖上的格子 | 角色 | 在程式裡 |
| --- | --- | --- |
| 從作業倉庫抓作業 | MCP（由**程式**呼叫，不需要判斷） | `抓作業` 節點呼叫 `list_homework`、`get_homework` |
| 一定先檢查語法 | 程式 | 語法錯就直接回，**不叫 AI** |
| 每份作業批改一次 | 同一套批改 agent（＝投影片 37 的回饋 agent）＋ Skill ＋ 腳本：批次，不是多開角色 | `Send` 把每份送進「批一份」；手冊 `friendly-feedback`，工具 `check_naming`、`run_code` |
| 檢查回饋有沒有亂講 | 驗證 agent（獨立檢查職責，程式裡叫「稽核員」）＋ 程式 | 程式先查行號，稽核員再看內容；不通過最多退回重寫一次；仍不通過會保留失敗狀態。稽核員只拿到作業原文與回饋，**沒有題目要求**，不能判斷是否誤解題意（投影片 35 的完整設計需要題目＋作業＋草稿） |
| 寄信 | MCP ＋ 人工確認 | `interrupt()` 停下來，人決定後才呼叫 `send_email`；選「都不要寄」就直接結束 |

表裡的兩個新寫法：
- `Send`：把清單中的每份資料送進「批一份」節點，可以同時處理。節點先查語法，通過才建立批改 agent。
- `interrupt()`：流程跑到這裡先停住、把 State 存起來，等人回覆之後再接著跑。

流程圖只有四站（抓作業 → 批一份 → 寄信前確認 → 寄信）；語法檢查、批改與驗證都寫在「批一份」節點裡，每份各跑一次（投影片 37 把它們畫成分開的格子）。選「都不要寄」時，流程從「寄信前確認」直接結束，不進「寄信」。

注意「不要三人接力」：一份作業從批改到修正都是**同一個批改員**；語意判斷交給另一個只負責檢查的 agent；獨立檢查仍可能漏錯。

### 🤔 預測題
1. 第一輪只選小美：預測哪個節點會停下來、怎樣確認還沒有寄信。第二輪整批有 5 份，其中小明語法錯，會有幾份需要模型批改？
2. 「同時批幾份」從 1 調到 3，會比較快嗎？token 會變少嗎？
3. 第 1 步跑完時，信寄出去了嗎？

In [ ]:
%%writefile /content/review-bot/deploy/multi_agent.py
"""2-E 練習 2：把第 2 堂的整張分工圖在 Colab 跑一次（多 agent ＋ MCP）。

START → 抓作業 ─Send×N→ 批一份 → 寄信前確認 ─要寄→ 寄信 → END
         (MCP)             │          (interrupt)  └都不要寄→ END   (MCP)
                           └ 程式檢查語法 → 批改 agent（手冊＋腳本＋run_code）→ 驗證 agent

對照分工圖中可以組合的五種角色（本版是簡化版：沒有題目附件與 read_spec）：
- 程式：外層流程的順序、語法檢查、「驗證不過就重寫一次」、選「都不要寄」就直接結束
- Tool：check_naming（手冊附的腳本）、run_code
- MCP ：作業倉庫的 list_homework / get_homework / send_email（工具住在另一個程式裡）
- Skill：friendly-feedback 手冊，當批改 agent 的 system prompt
- agent：同一套批改 agent 逐份處理（Send 分派，每份一張乾淨的桌子）＋ 一個職責獨立的驗證 agent
"""

import json
import operator
import re
import sys
from pathlib import Path
from typing import Annotated, TypedDict

from langchain.agents import create_agent
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send, interrupt

from graph import check_syntax
from skill_loader import 列出名片, 讀內文
from tools.check_naming import check_naming
from tools.run_code import run_code

MCP_SERVER = Path(__file__).resolve().with_name("homework_mcp.py")
存檔位置 = Path(__file__).resolve().with_name("multi_agent.sqlite")
寄信選項 = ["全部寄出", "只寄驗證通過的", "都不要寄"]


# ---------- MCP：作業倉庫的插座 ----------

def 作業倉庫設定(網址: str | None = None) -> dict:
    """給網址就連遠端（Streamable HTTP），沒給就在本機開一個（stdio）。"""
    if 網址:
        return {"作業倉庫": {"url": 網址, "transport": "streamable_http"}}
    return {"作業倉庫": {"command": sys.executable, "args": [str(MCP_SERVER)], "transport": "stdio"}}


async def 取得MCP工具(設定: dict) -> dict:
    return {t.name: t for t in await MultiServerMCPClient(設定).get_tools()}


def 結果文字(結果) -> str:
    """MCP 工具回傳的是一串內容區塊，取出裡面的文字。"""
    if isinstance(結果, str):
        return 結果
    if isinstance(結果, list):
        return "".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in 結果)
    return str(結果)


# ---------- State ----------

class 總State(TypedDict, total=False):
    名單: list[dict]                             # [{"學生", "email", "code"}]
    結果們: Annotated[list[dict], operator.add]   # 每個「批一份」交回來的結果，往下接著寫
    決定: str
    寄件紀錄: list[str]


class 一份State(TypedDict):
    學生: str
    email: str
    code: str


# ---------- 驗證 agent：獨立檢查作業與回饋，不負責撰寫回饋 ----------

稽核員人設 = (
    "你是嚴格的稽核員，只負責檢查別人寫的作業回饋，自己不寫回饋。\n"
    "檢查三件事：1. 回饋說的問題在作業裡真的存在嗎 2. 有沒有亂講行號 3. 有沒有用「錯誤」「很差」這類字眼。\n"
    '只回一行 JSON，不要其他文字：{"通過": true 或 false, "原因": "一句話"}'
)


def 驗證(model, code: str, feedback: str) -> dict:
    # 程式能判斷的，先用程式檢查（這幾項不花 token）；稽核員每份都會再呼叫一次模型
    問題 = []
    if "優點" not in feedback or "改進" not in feedback:
        問題.append("少了「優點」或「改進」段落")
    行數 = len(code.splitlines())
    for n in re.findall(r"第\s*(\d+)\s*行", feedback):
        if not 1 <= int(n) <= 行數:
            問題.append(f"提到第 {n} 行，但作業只有 {行數} 行")
    # 程式判斷不了的，交給稽核員
    帶行號 = "\n".join(f"{i:>2} | {line}" for i, line in enumerate(code.splitlines(), 1))
    回答 = model.invoke([SystemMessage(稽核員人設), HumanMessage(f"作業：\n{帶行號}\n\n回饋：\n{feedback}")]).text
    m = re.search(r"\{.*\}", 回答, re.S)
    try:
        判斷 = json.loads(m.group(0)) if m else {}
    except json.JSONDecodeError:
        判斷 = {}
    if not isinstance(判斷, dict) or not isinstance(判斷.get("通過"), bool):
        問題.append("稽核結果無法解析成布林通過值，請人工檢查")
    elif not 判斷["通過"]:
        問題.append("稽核員：" + str(判斷.get("原因", 回答.strip()[:60])))
    return {"通過": not 問題, "原因": "；".join(問題) or "OK"}


# ---------- 把零件接起來 ----------

def 建立多agent小助教(model, mcp設定: dict | None = None, checkpointer=None, 只批學生: str | None = None):
    mcp設定 = mcp設定 or 作業倉庫設定()
    手冊 = 讀內文(列出名片(["friendly-feedback"])[0])
    批改員人設 = (
        f"{手冊}\n\n"
        "# 你的工具\n"
        "- check_naming：檢查變數命名（手冊附的腳本），要談命名時用它\n"
        "- run_code：執行作業看輸出\n"
        "要不要用、用哪一個，你自己決定。回饋不超過 250 字。"
    )

    async def 抓作業(state: 總State) -> dict:
        # 抓作業不需要判斷，所以是「程式」呼叫 MCP 工具，不是 agent
        tools = await 取得MCP工具(mcp設定)
        名單 = json.loads(結果文字(await tools["list_homework"].ainvoke({})))
        if 只批學生 is not None:
            名單 = [a for a in 名單 if a["學生"] == 只批學生]
            if not 名單:
                raise ValueError(f"作業倉庫沒有學生：{只批學生}")
        for a in 名單:
            a["code"] = 結果文字(await tools["get_homework"].ainvoke({"student": a["學生"]}))
        return {"名單": 名單}

    def 分派(state: 總State) -> list[Send]:
        # 每份作業開一個「批一份」：各自一張乾淨的桌子
        return [Send("批一份", a) for a in state["名單"]]

    def 批一份(state: 一份State) -> dict:
        code = state["code"]
        語法 = check_syntax({"code": code})
        if not 語法["syntax_ok"]:   # 能寫成 if 的事：程式直接回，不叫 AI
            feedback = f"語法：{語法['error']}\n改進：先把這一行修好，再執行一次看看。"
            return {"結果們": [{**state, "feedback": feedback, "用到的工具": [], "重寫次數": 0,
                              "驗證": {"通過": True, "原因": "語法錯誤由程式直接回覆，沒有經過 AI"}}]}
        批改員 = create_agent(model, tools=[check_naming, run_code], system_prompt=批改員人設)
        messages = [{"role": "user", "content": f"請批改這份作業：\n```python\n{code}\n```"}]
        用到的工具, 重寫次數 = [], 0
        while True:
            result = 批改員.invoke({"messages": messages}, config={"recursion_limit": 25})
            # result 含前一輪完整紀錄；只取本輪新增訊息，避免重複計入舊工具要求。
            用到的工具 += [c["name"] for m in result["messages"][len(messages):] for c in (getattr(m, "tool_calls", None) or [])]
            feedback = result["messages"][-1].text.strip()
            結論 = 驗證(model, code, feedback)
            if 結論["通過"] or 重寫次數 >= 1:      # 驗證不過最多退回重寫一次
                break
            重寫次數 += 1
            messages = result["messages"] + [{"role": "user", "content": f"稽核員退回：{結論['原因']}。請修正後重寫回饋。"}]
        return {"結果們": [{**state, "feedback": feedback, "用到的工具": 用到的工具,
                          "重寫次數": 重寫次數, "驗證": 結論}]}

    def 寄信前確認(state: 總State) -> dict:
        # 寄信不可逆：停下來，把要寄的內容交給人看
        清單 = [{"學生": r["學生"], "email": r["email"], "驗證通過": r["驗證"]["通過"],
                "回饋": r["feedback"]} for r in state["結果們"]]
        決定 = interrupt({"要寄出的信": 清單, "選項": 寄信選項})
        if 決定 == "都不要寄":
            return {"決定": 決定, "寄件紀錄": ["沒有寄出任何信（決定：都不要寄）"]}
        return {"決定": 決定}

    def 要不要寄(state: 總State) -> str:
        # 選「都不要寄」就直接結束，不進寄信節點
        return END if state.get("決定") == "都不要寄" else "寄信"

    async def 寄信(state: 總State) -> dict:
        要寄 = [r for r in state["結果們"]
              if state.get("決定") == "全部寄出" or (state.get("決定") == "只寄驗證通過的" and r["驗證"]["通過"])]
        if not 要寄:
            return {"寄件紀錄": [f"沒有寄出任何信（決定：{state.get('決定')}）"]}
        tools = await 取得MCP工具(mcp設定)
        紀錄 = []
        for r in 要寄:
            結果 = await tools["send_email"].ainvoke({"to": r["email"], "subject": "作業回饋", "body": r["feedback"]})
            紀錄.append(結果文字(結果))
        return {"寄件紀錄": 紀錄}

    builder = StateGraph(總State)
    builder.add_node("抓作業", 抓作業)
    builder.add_node("批一份", 批一份)
    builder.add_node("寄信前確認", 寄信前確認)
    builder.add_node("寄信", 寄信)
    builder.add_edge(START, "抓作業")
    builder.add_conditional_edges("抓作業", 分派, ["批一份"])
    builder.add_edge("批一份", "寄信前確認")
    builder.add_conditional_edges("寄信前確認", 要不要寄, ["寄信", END])
    builder.add_edge("寄信", END)
    return builder.compile(checkpointer=checkpointer)


async def 開存檔(path: Path = 存檔位置):
    """interrupt 要等人確認，所以 State 一定要存起來；這裡存成 SQLite 檔（雲端改用 Postgres）。"""
    import aiosqlite
    from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

    return AsyncSqliteSaver(await aiosqlite.connect(str(path)))


class 用量計(BaseCallbackHandler):
    """數模型被呼叫幾次、加總 token（Colab 內建模型的 token 是估算值）。"""

    def __init__(self):
        self.次數 = 0
        self.input_tokens = 0
        self.output_tokens = 0

    def on_llm_end(self, response, **kwargs):
        self.次數 += 1
        for generations in response.generations:
            for g in generations:
                usage = getattr(getattr(g, "message", None), "usage_metadata", None) or {}
                self.input_tokens += usage.get("input_tokens", 0)
                self.output_tokens += usage.get("output_tokens", 0)


In [6]:
#@title ▶ 執行（第 1 步）：整張分工圖跑一次，停在寄信前面
thread_id = "期中-第一批"  #@param {type:"string"}
批改範圍 = "先批小美一份"  #@param ["先批小美一份", "整批五份"]
同時批幾份 = 1  #@param {type:"slider", min:1, max:5, step:1}
作業倉庫接法 = "本機插座 stdio"  #@param ["本機插座 stdio", "網址 Streamable HTTP（要先跑練習 1 第二格）"]
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
import time
from deploy import multi_agent

網址 = None
if 作業倉庫接法.startswith("網址"):
    port檔 = BASE / "deploy" / ".mcp_port"
    if not port檔.exists():
        raise SystemExit("⚠️ 還沒用網址開 MCP server：先執行練習 1 的第二格，或改選「本機插座 stdio」")
    網址 = f"http://127.0.0.1:{port檔.read_text()}/mcp"

存檔 = await multi_agent.開存檔()
小助教 = multi_agent.建立多agent小助教(model, mcp設定=multi_agent.作業倉庫設定(網址), checkpointer=存檔,
                                    只批學生="小美" if 批改範圍 == "先批小美一份" else None)
config = {"configurable": {"thread_id": thread_id}, "max_concurrency": 同時批幾份}
if (await 小助教.aget_state(config)).values:
    await 存檔.conn.close()
    raise SystemExit(f"⚠️ thread_id「{thread_id}」已經用過了：換一個新的（例如「期中-第二批」），或直接去跑第 2 步")
顯示流程圖(小助教)

用量 = multi_agent.用量計()
t0 = time.time()
結果 = await 小助教.ainvoke({"結果們": []}, {**config, "callbacks": [用量]})
秒數 = time.time() - t0
for r in 結果["結果們"]:
    驗證 = "✅ 通過" if r["驗證"]["通過"] else "❌ " + r["驗證"]["原因"]
    print(f"【{r['學生']}】工具：{'、'.join(r['用到的工具']) or '（沒有要求工具；不等於沒叫 AI）'}｜驗證：{驗證}｜退回重寫 {r['重寫次數']} 次")
    print("回饋全文：\n" + r["feedback"])
    print("驗證原因：", r["驗證"]["原因"])
print(f"\n⏱️ {秒數:.0f} 秒（同時最多 {同時批幾份} 份）；模型呼叫 {用量.次數} 次，約 {用量.input_tokens + 用量.output_tokens:,} token（估算）")

if "__interrupt__" in 結果:
    print("\n⏸️  停在寄信前面，等人確認。準備寄出的信：")
    for 信 in 結果["__interrupt__"][0].value["要寄出的信"]:
        print(f"    → {信['學生']} <{信['email']}>（驗證{'通過' if 信['驗證通過'] else '沒過'}）")
        print(信["回饋"])
print("存檔裡的下一站：", (await 小助教.aget_state(config)).next)
await 存檔.conn.close()
記下(f"2-E 練習 2：整張分工圖（thread_id={thread_id}）",
     {"結果": [{k: r[k] for k in ("學生", "用到的工具", "驗證", "重寫次數")} for r in 結果["結果們"]],
      "模型呼叫次數": 用量.次數, "token": 用量.input_tokens + 用量.output_tokens, "秒數": round(秒數)})

（畫圖服務暫時連不上，改用文字版流程圖；可以貼到 https://mermaid.live 看圖）

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	\6293\4f5c\696d(抓作業)
	\6279\4e00\4efd(批一份)
	\5bc4\4fe1\524d\78ba\8a8d(寄信前確認)
	\5bc4\4fe1(寄信)
	__end__([<p>__end__</p>]):::last
	__start__ --> \6293\4f5c\696d;
	\5bc4\4fe1\524d\78ba\8a8d --> \5bc4\4fe1;
	\6279\4e00\4efd --> \5bc4\4fe1\524d\78ba\8a8d;
	\6293\4f5c\696d -.-> \6279\4e00\4efd;
	\5bc4\4fe1 --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

【小美】工具：（沒有要求工具；不等於沒叫 AI）｜驗證：✅ 通過｜退回重寫 0 次
回饋全文：
優點：函式結構定義得非常清楚，能運用參數進行數值加法，並且在最後有主動呼叫函式來印出結果，測試的習慣相當棒。

改進：函式中將相加結果存放在 total 變數，但在回傳時寫成了 summ，導致程式執行時找不到該變數。建議將第 3 行的 `return summ` 改為 `return total`，這樣就能順利輸出相加後的數值 5 了。
驗證原因： OK

⏱️ 25 秒（同時最多 1 份）；模型呼叫 2 次，約 1,034 token（估算）

⏸️  停在寄信前面，等人確認。準備寄出的信：
    → 小美 <xiaomei@example.com>（驗證通過）
優點：函式結構定義得非常清楚，能運用參數進行數值加法，並且在最後有主動呼叫函式來印出結果，測試的習慣相當棒。

改進：函式中將相加結果存放在 total 變數，但在回傳時寫成了 summ，導致

In [7]:
#@title ▶ 執行（第 2 步）：人看過了，決定要不要寄
thread_id = "期中-第一批"  #@param {type:"string"}
決定 = "只寄驗證通過的"  #@param ["只寄驗證通過的", "全部寄出", "都不要寄"]
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
from langgraph.types import Command
from deploy import multi_agent

存檔 = await multi_agent.開存檔()
小助教 = multi_agent.建立多agent小助教(model, checkpointer=存檔)
config = {"configurable": {"thread_id": thread_id}}
if (await 小助教.aget_state(config)).next != ("寄信前確認",):
    print(f"thread_id「{thread_id}」沒有在等確認的信：先執行第 1 步，或檢查 thread_id 有沒有打錯。")
else:
    結果 = await 小助教.ainvoke(Command(resume=決定), config)   # resume 的值就是 interrupt() 的回傳值
    for 紀錄 in 結果["寄件紀錄"]:
        print("📮", 紀錄)
    寄件匣 = BASE / "寄件匣.txt"
    if 寄件匣.exists():
        print("\n寄件匣.txt 最後幾行：")
        print("\n".join(寄件匣.read_text(encoding="utf-8").splitlines()[-6:]))
    記下(f"2-E 練習 2：決定「{決定}」（thread_id={thread_id}）", {"寄件紀錄": 結果["寄件紀錄"]})
await 存檔.conn.close()

📮 （模擬）已寄給 xiaomei@example.com

寄件匣.txt 最後幾行：
[03:26:43] 寄給 xiaomei@example.com｜作業回饋
優點：函式結構定義得非常清楚，能運用參數進行數值加法，並且在最後有主動呼叫函式來印出結果，測試的習慣相當棒。

改進：函式中將相加結果存放在 total 變數，但在回傳時寫成了 summ，導致程式執行時找不到該變數。建議將第 3 行的 `return summ` 改為 `return total`，這樣就能順利輸出相加後的數值 5 了。



### 👀 你看到了什麼

- **第一輪完成檢查**：應看到小美的一份回饋、驗證結果、待確認的信與下一站。第 1 步只準備信；第 2 步用相同 thread_id 決定後，才寫入假的寄件匣。
- 第二輪：把範圍改成「整批五份」、兩步都換成新編號「期中-第二批」、同時批幾份設為 1。小明語法錯由程式直接回覆；其餘四份各自進入批改與驗證，退回重寫時會再驗證。
- 第三輪選做：再換新編號，保持整批五份、同時批幾份改成 3，記錄秒數、模型呼叫數與 token，和第二輪比較。**並行不保證更快，也不保證 token 相同**；模型回答、重寫次數、限流和系統負載都會影響結果。
- 每個批改員自行選工具；驗證 agent 的價值是獨立檢查職責與輸入，不只是換個人設。驗證通過仍不代表內容絕對正確；重寫後未通過的回饋仍會列出，預設「只寄驗證通過的」會排除它們。小明的語法回覆由程式直接產生，沒有交給驗證 agent，輸出會註明原因。
- 想從頭再做請用新 thread_id；已用過的編號不會清空。先能說明「待確認 → 人決定 → 寄件紀錄」，再進入 API 練習。

---
## 練習 3（進階選做）：包成 API 服務

> 📍 **對應簡報**：投影片 48（API 那一格）、49「離開 Colab：三樣東西要換掉」、51「選讀：部署方式，依維運需求選擇」、54「寄信前等待確認：interrupt」

練習 2 是你在 notebook 裡按 ▶ 跑的。上雲端之後，沒有人會按 notebook 的 ▶：幹部是在後台網頁按「開始批改」和「確認寄信」，
網頁再用 **HTTP**（瀏覽器開網頁用的同一套規則）呼叫你的服務。這種「讓別的程式用網址叫你做事」的介面，叫做 **API**。
下一格寫出 `deploy/server.py`，用 FastAPI（寫 API 的 Python 套件）開了四個網址：

| 網址 | 做什麼 |
| --- | --- |
| `GET /health` | 雲端平台定時敲它，確認服務還活著 |
| `POST /batch` | 開始一批：抓作業、批改、驗證，停在寄信前面，回傳待確認的信 |
| `POST /batch/{thread_id}/decision` | 人確認之後：寄或不寄 |
| `GET /threads/{thread_id}` | 查一批的進度 |

### 🤔 預測題
「開始批改」和「確認寄信」是兩個不同的請求，中間可能隔三天。第二個請求怎麼知道要寄哪些信？

In [8]:
%%writefile /content/review-bot/deploy/server.py
"""2-E 練習 3：把多 agent 小助教包成一個網路服務（API）。

上雲端之後，雲端機器上真正在跑的就是這種程式：
幹部的後台網頁用 HTTP 叫它「開始批改」，它把待寄的信列出來；幹部按下確認，再用 HTTP 叫它「寄」。
每一批用一個 thread_id 記住進度：開始和確認是兩個不同的請求，中間 State 存在存檔裡。

本機或容器裡啟動：uvicorn deploy.server:app --port 8080
作業倉庫 MCP：有設環境變數 HOMEWORK_MCP_URL 就連那個網址，沒設就在本機開一個（stdio）。
"""

import os

from fastapi import FastAPI
from langgraph.types import Command
from pydantic import BaseModel

from deploy import multi_agent
from llm import get_model

app = FastAPI(title="作業小助教 API")
_小助教 = None


async def 小助教():
    """第一次有人呼叫時才建立（模型、MCP 設定、存檔都只建一次）。"""
    global _小助教
    if _小助教 is None:
        _小助教 = multi_agent.建立多agent小助教(
            get_model(),
            mcp設定=multi_agent.作業倉庫設定(os.environ.get("HOMEWORK_MCP_URL")),
            checkpointer=await multi_agent.開存檔(),
        )
    return _小助教


class 批改請求(BaseModel):
    thread_id: str
    同時批幾份: int = 2


class 決定請求(BaseModel):
    決定: str   # 全部寄出／只寄驗證通過的／都不要寄


@app.get("/health")
def health() -> dict:
    """雲端平台會定時敲這個網址，確認服務還活著。"""
    return {"ok": True}


@app.post("/batch")
async def 開始批改(req: 批改請求) -> dict:
    """從作業倉庫抓作業、平行批改、驗證，然後停在寄信前面等人確認。"""
    config = {"configurable": {"thread_id": req.thread_id}, "max_concurrency": req.同時批幾份}
    圖 = await 小助教()
    if (await 圖.aget_state(config)).values:
        return {"thread_id": req.thread_id, "錯誤": "此編號已有存檔；請查進度或用新編號開始，不會重新批改或追加結果。"}
    結果 = await 圖.ainvoke({"結果們": []}, config)
    待確認 = 結果["__interrupt__"][0].value if "__interrupt__" in 結果 else None
    return {"thread_id": req.thread_id, "待確認": 待確認}


@app.post("/batch/{thread_id}/decision")
async def 確認寄信(thread_id: str, req: 決定請求) -> dict:
    """人看過之後的決定：帶著同一個 thread_id 接著跑。"""
    config = {"configurable": {"thread_id": thread_id}}
    圖 = await 小助教()
    if req.決定 not in multi_agent.寄信選項:
        return {"thread_id": thread_id, "錯誤": "請選擇有效的寄信決定"}
    if (await 圖.aget_state(config)).next != ("寄信前確認",):
        return {"thread_id": thread_id, "錯誤": "這個 thread_id 沒有在等確認的信；請先查進度"}
    結果 = await 圖.ainvoke(Command(resume=req.決定), config)
    return {"thread_id": thread_id, "寄件紀錄": 結果.get("寄件紀錄", [])}


@app.get("/threads/{thread_id}")
async def 查進度(thread_id: str) -> dict:
    """查某個 thread_id 的存檔：State 有沒有保留下來、停在哪一站。"""
    snap = await (await 小助教()).aget_state({"configurable": {"thread_id": thread_id}})
    v = snap.values
    return {
        "thread_id": thread_id,
        "有存檔": bool(v),
        "下一站": list(snap.next),
        "批好幾份": len(v.get("結果們", [])),
        "決定": v.get("決定"),
        "寄件紀錄": v.get("寄件紀錄"),
    }


Writing /content/review-bot/deploy/server.py


In [9]:
#@title ▶ 執行：在 Colab 裡把小助教開成 API 服務
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
import socket, threading, time
import requests, uvicorn
from deploy import server

舊的 = getattr(sys, "_小助教伺服器", None)       # 重跑這一格時，先把上一次開的關掉
if 舊的:
    舊的.should_exit = True
    time.sleep(1)
with socket.socket() as s:
    s.bind(("127.0.0.1", 0))
    port = s.getsockname()[1]
伺服器 = uvicorn.Server(uvicorn.Config(server.app, host="127.0.0.1", port=port, log_level="warning"))
threading.Thread(target=伺服器.run, daemon=True).start()
sys._小助教伺服器 = 伺服器
for _ in range(50):
    try:
        if requests.get(f"http://127.0.0.1:{port}/health", timeout=1).ok:
            break
    except requests.ConnectionError:
        time.sleep(0.2)
else:
    raise SystemExit("⚠️ API 未通過健康檢查，請查看上方錯誤並重新啟動服務。")
(BASE / "deploy" / ".port").write_text(str(port))
print(f"✅ 小助教 API 開好了：http://127.0.0.1:{port}")
for 網址 in ["GET  /health", "POST /batch", "POST /batch/{thread_id}/decision", "GET  /threads/{thread_id}"]:
    print("   ", 網址)

✅ 小助教 API 開好了：http://127.0.0.1:42257
    GET  /health
    POST /batch
    POST /batch/{thread_id}/decision
    GET  /threads/{thread_id}


In [10]:
#@title ▶ API 第 1 步：開始一批、閱讀待寄內容與進度
thread_id = "API-第一批"  #@param {type:"string"}
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
import json, requests

port檔 = BASE / "deploy" / ".port"
if not port檔.exists():
    raise SystemExit("⚠️ 服務還沒開：先執行上一格")
網址 = f"http://127.0.0.1:{port檔.read_text()}"
try:
    開始 = requests.post(f"{網址}/batch", json={"thread_id": thread_id, "同時批幾份": 2}, timeout=900).json()
except requests.ConnectionError:
    raise SystemExit("⚠️ 服務沒有在跑（可能重新啟動過執行階段）：先執行上一格")

if 開始.get("錯誤"):
    raise SystemExit(開始["錯誤"] + " 若要確認既有信件，請到下一格使用相同編號。")
待寄 = (開始.get("待確認") or {}).get("要寄出的信", [])
print(f"POST /batch → 停在寄信前面，待確認 {len(待寄)} 封：")
for 信 in 待寄:
    print(f"    {信['學生']} <{信['email']}> 驗證{'通過' if 信['驗證通過'] else '沒過'}")
    print(信["回饋"])
print(f"\nGET /threads/{thread_id} →", json.dumps(requests.get(f"{網址}/threads/{thread_id}", timeout=30).json(), ensure_ascii=False))

print("請先閱讀待寄內容，再到下一格選擇寄信決定；本格尚未寄信。")
記下(f"2-E 練習 3：API 開始（thread_id={thread_id}）", 開始)

POST /batch → 停在寄信前面，待確認 5 封：
    小明 <xiaoming@example.com> 驗證通過
語法：第 1 行少了右括號 )
改進：先把這一行修好，再執行一次看看。
    小華 <xiaohua@example.com> 驗證通過
優點：函式結構清晰簡潔，參數與函式命名皆符合 Python 慣例，能順利完成兩數相加並輸出正確結果。

改進：若想讓程式碼更完整且易於維護，可以參考以下調整：
1. 加入型別提示（Type Hints）：建議改寫為 `def add(a: int, b: int) -> int:`，讓閱讀程式碼的人能一眼看出預期的參數與回傳型別。
2. 補充函式說明（Docstring）：可以在函式內加上說明文字，例如 `"""計算並回傳兩數的總和。"""`，有助於提升整體的可讀性。
    小美 <xiaomei@example.com> 驗證通過
優點：函式架構非常清晰，參數設計與相加邏輯都很明確，整體寫法乾淨好讀。

改進：在回傳值的部分，目前寫的是未定義的 `summ`，建議可以調整為上面算好的變數 `total`（也就是改為 `return total`），這樣執行時就能順利印出相加後的結果囉！
    阿傑 <ajie@example.com> 驗證通過
優點：程式邏輯非常清晰易懂，運用迴圈累加並計算平均值的流程很順暢，能順利得到正確的計算結果。

改進：
1. 命名風格：Python 社群習慣使用小寫搭配底線的蛇形命名法（snake_case）。建議可將函式名稱 `CalcAverage` 改為 `calc_average`，變數 `scoreList` 改為 `score_list`。
2. 邊界情況防護：若傳入的串列沒有元素，執行除法時會中斷。建議在開頭加入判斷，例如 `if not scoreList: return 0`，能讓函式更加穩健。
    小芳 <xiaofang@example.com> 驗證通過
優點：程式邏輯非常清晰，運用迴圈累加並正確計算出平均值，函式與變數的命名也很貼切好懂。

改進：
1. 縮排一致性：目前每行的縮排空格數不太一致（包含 2 格與 8 格）。建議依照 Python 規範統一使用 4 個空格，例如將 `total += s` 縮排為 4 個空格，排版會更整齊。

**停一下**：上一格只是開始批改與列出待寄內容。讀完後，再在下一格填入同一個 thread_id 並選擇決定。
本範例仍只寫假的寄件匣；分成兩格是為了保留人實際閱讀與決定的時間。

In [11]:
#@title ▶ API 第 2 步：人讀完後，確認寄信並查進度
thread_id = "API-第一批"  #@param {type:"string"}
決定 = "只寄驗證通過的"  #@param ["只寄驗證通過的", "全部寄出", "都不要寄"]
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
import json, requests

port檔 = BASE / "deploy" / ".port"
if not port檔.exists():
    raise SystemExit("請先啟動 API 服務並完成 API 第 1 步。")
網址 = f"http://127.0.0.1:{port檔.read_text()}"
確認 = requests.post(f"{網址}/batch/{thread_id}/decision", json={"決定": 決定}, timeout=300).json()
print(f"\nPOST /batch/{thread_id}/decision（{決定}）→")
for 紀錄 in 確認.get("寄件紀錄") or [確認.get("錯誤")]:
    print("    📮", 紀錄)
print(f"\nGET /threads/{thread_id} →", json.dumps(requests.get(f"{網址}/threads/{thread_id}", timeout=30).json(), ensure_ascii=False))
print("GET /threads/還沒開始的那批 →", json.dumps(requests.get(f"{網址}/threads/還沒開始的那批", timeout=30).json(), ensure_ascii=False))
記下(f"2-E 練習 3：API 服務（thread_id={thread_id}）", {"確認": 確認})


POST /batch/API-第一批/decision（只寄驗證通過的）→
    📮 （模擬）已寄給 xiaoming@example.com
    📮 （模擬）已寄給 xiaohua@example.com
    📮 （模擬）已寄給 xiaomei@example.com
    📮 （模擬）已寄給 ajie@example.com
    📮 （模擬）已寄給 xiaofang@example.com

GET /threads/API-第一批 → {"thread_id": "API-第一批", "有存檔": true, "下一站": [], "批好幾份": 5, "決定": "只寄驗證通過的", "寄件紀錄": ["（模擬）已寄給 xiaoming@example.com", "（模擬）已寄給 xiaohua@example.com", "（模擬）已寄給 xiaomei@example.com", "（模擬）已寄給 ajie@example.com", "（模擬）已寄給 xiaofang@example.com"]}
GET /threads/還沒開始的那批 → {"thread_id": "還沒開始的那批", "有存檔": false, "下一站": [], "批好幾份": 0, "決定": null, "寄件紀錄": null}


### 👀 你看到了什麼

- 兩個請求之間靠 **thread_id** 接起來：第一個請求跑到 `interrupt()` 就把 State 存進存檔、回應「待確認」，此時尚未寄信；
  第二個請求帶同一個 thread_id，依決定寄或不寄。還沒開始的那批查不到任何存檔；已用過的編號不能再次開始批改，以免把結果混在一起。
- 雲端機器隨時會被換掉，所以上線時存檔要放在資料庫（Postgres），不能放在這台機器的 SQLite 檔。
- 真的上線時，`POST /batch` 跑 200 份會很久，通常改成「丟進任務佇列、馬上回應」，LangSmith Deployment 這類平台內建這件事。
- `server.py` 是「自己的容器」這條路真正在跑的 API（`Dockerfile` 啟動 `uvicorn deploy.server:app`）。代管平台改讀 `langgraph.json` 指向的 `deploy/app_graph.py:graph`，用平台內建的 threads／runs API，沒有 `POST /batch`。

---
## 練習 4（進階選做）：上線前檢查，順便產生部署檔

> 📍 **對應簡報**：投影片 49（金鑰放 Secret Manager）、50（langgraph.json 是出貨單）、55「上線之後：看得見、關得住」

最後一步：真的要上雲端之前，先用程式檢查一遍，再把出貨要用的檔案準備好。
下面幾格會寫出上雲端要用的檔案（**只產生檔案，不會真的打包或部署**）：

| 檔案 | 是什麼 |
| --- | --- |
| `langgraph.json` | 出貨單：告訴 LangSmith Deployment / `langgraph dev`「graph 在哪個檔案的哪個變數」 |
| `deploy/app_graph.py` | 出貨單指向的進入點：模型金鑰、遠端 MCP 網址都從環境變數讀 |
| `requirements.txt` | 要裝哪些套件（限制相容版本範圍；`mcp` 和 `langchain-mcp-adapters` 一定要一起釘） |
| `Dockerfile` | 把程式、`skills/`、`scripts/` 裝進同一個容器的食譜 |
| `.dockerignore` | 不准裝進容器的東西：`.env`、存檔、紀錄 |
| `deploy/preflight.py` | 上線前檢查：每一項都用程式檢查，不靠感覺 |

`langgraph.json` 會讀 `.env`（只在自己電腦上用、不放進 git）。這本不產生 `.env`；要用 `langgraph dev` 時自己建立，範本在 repo 的 `review-bot/.env.example`。

### 🤔 預測題
上線前檢查有 10 項，你猜哪幾項會是 ⚠️（上雲端前還要處理）？

In [12]:
%%writefile /content/review-bot/langgraph.json
{
  "dependencies": ["."],
  "graphs": {
    "review_bot": "./deploy/app_graph.py:graph"
  },
  "env": ".env"
}


Writing /content/review-bot/langgraph.json


In [13]:
%%writefile /content/review-bot/deploy/app_graph.py
"""langgraph.json 指向的進入點：LangSmith Deployment / `langgraph dev` 會 import 這個檔案裡的 graph。

部署平台會自己提供 checkpointer（例如 Postgres），所以這裡不用 開存檔()。
- 模型：WORKSHOP_PROVIDER=gemini，金鑰 GOOGLE_API_KEY 由平台的 Secret 注入
- 作業倉庫 MCP：HOMEWORK_MCP_URL 指向遠端 MCP server 的網址（沒設就在同一台機器上用 stdio 開一個）
"""

import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
os.environ.setdefault("WORKSHOP_PROVIDER", "gemini")

from deploy.multi_agent import 作業倉庫設定, 建立多agent小助教  # noqa: E402
from llm import get_model  # noqa: E402

graph = 建立多agent小助教(get_model(), mcp設定=作業倉庫設定(os.environ.get("HOMEWORK_MCP_URL")))


Writing /content/review-bot/deploy/app_graph.py


In [14]:
%%writefile /content/review-bot/requirements.txt
# 作業小助教上雲端需要的套件（版本範圍和 notebook 一致）
langgraph>=1.1,<2
langchain>=1.2,<2
langchain-google-genai>=4,<5
langgraph-checkpoint-sqlite>=3,<4
# MCP：兩個要一起釘，版本不配會 import 失敗
mcp>=1.20,<2
langchain-mcp-adapters>=0.3,<0.4
fastapi>=0.115
uvicorn>=0.30
# 正式環境把 SQLite 換成 Postgres 時再加：langgraph-checkpoint-postgres


Writing /content/review-bot/requirements.txt


In [15]:
%%writefile /content/review-bot/Dockerfile
# 把作業小助教（程式 ＋ skills/ ＋ scripts/）裝進同一個容器
# 在 review-bot/ 資料夾裡：
#   docker build -t review-bot .
#   docker run -p 8080:8080 -e GOOGLE_API_KEY=你的key -e HOMEWORK_MCP_URL=https://遠端MCP網址/mcp review-bot
# 金鑰不要寫進這個檔案：部署時由雲端的 Secret Manager 變成環境變數注入
# 沒設 HOMEWORK_MCP_URL 的話，作業倉庫 MCP 會在同一個容器裡用 stdio 開起來（示範用）
FROM python:3.12-slim

WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# 程式、手冊、腳本一起進映像：手冊和程式永遠是同一個版本
COPY . .

ENV WORKSHOP_PROVIDER=gemini \
    PORT=8080
CMD ["sh", "-c", "uvicorn deploy.server:app --host 0.0.0.0 --port ${PORT}"]


Writing /content/review-bot/Dockerfile


In [16]:
%%writefile /content/review-bot/.dockerignore
# 不要裝進容器的東西：金鑰、本機存檔、執行紀錄
.env
.llm.json
checkpoints.sqlite*
deploy/*.sqlite*
deploy/.port
deploy/.mcp_port
deploy/*.log
history.jsonl
寄件匣.txt
2B_results.json
__pycache__/


Writing /content/review-bot/.dockerignore


In [17]:
%%writefile /content/review-bot/deploy/preflight.py
"""練習 4：上線前檢查。每一項都是用程式檢查（sensor），不是靠感覺。

用法：python -m deploy.preflight（在 review-bot/ 資料夾裡）
"""

import os
import re
import sys
from pathlib import Path

BASE = Path(__file__).resolve().parent.parent

金鑰樣式 = [
    (r"AIza[0-9A-Za-z\-_]{35}", "Google API key"),
    (r"sk-[A-Za-z0-9_\-]{20,}", "OpenAI / Anthropic 類型的 key"),
    (r"(?i)(api[_-]?key|token|password)\s*=\s*['\"][^'\"\s]{8,}['\"]", "寫死在程式裡的密碼"),
]
可疑樣式 = [
    (r"curl[^\n|]*\|\s*(ba)?sh|wget[^\n|]*\|\s*(ba)?sh", "下載後直接執行（curl … | bash）"),
    (r"base64\s+(-d|--decode)", "解碼一段看不懂的內容"),
    (r"(?i)ignore (all |the )?previous|忽略(之前|以上|前面)", "要 AI 忽略之前的指示"),
    (r"(?i)環境變數|read_env|WORKSHOP_API_KEY|\$[A-Z_]*KEY", "要 AI 去讀金鑰或環境變數"),
]
文字檔 = {".py", ".md", ".sh", ".json", ".txt", ".toml", ".yaml", ".yml", ""}
略過 = {"__pycache__", ".llm.json", "history.jsonl", ".env.example"}


def _檔案們(root: Path):
    for p in sorted(root.rglob("*")):
        if p.is_file() and p.suffix in 文字檔 and not (set(p.parts) & 略過) and p.name not in 略過:
            yield p


def _掃(root: Path, 樣式) -> list[str]:
    找到 = []
    for p in _檔案們(root):
        if p.resolve() == Path(__file__).resolve():
            continue  # 不掃自己（這個檔案裡本來就寫著這些樣式）
        for i, line in enumerate(p.read_text(encoding="utf-8", errors="ignore").splitlines(), 1):
            for pattern, 說明 in 樣式:
                if re.search(pattern, line):
                    找到.append(f"{p.relative_to(BASE)} 第 {i} 行：{說明}")
    return 找到


def 檢查() -> list[tuple[str, str, str]]:
    """回傳 [(結果, 項目, 說明)]，結果是 ✅ / ⚠️ / ❌。"""
    結果 = []

    外洩 = _掃(BASE, 金鑰樣式)
    結果.append(("❌" if 外洩 else "✅", "程式和手冊裡沒有寫死的金鑰",
               "；".join(外洩) if 外洩 else "金鑰應該放在 Secrets / Secret Manager，從環境變數讀"))

    來源 = os.environ.get("WORKSHOP_PROVIDER") or "（看 0. 環境設定 的選擇）"
    有key = bool(os.environ.get("GOOGLE_API_KEY"))
    結果.append(("✅" if 有key else "⚠️", "模型金鑰從環境變數來",
               f"WORKSHOP_PROVIDER={來源}；GOOGLE_API_KEY {'已設定' if 有key else '沒設定（Colab 內建模型不需要，上雲端一定要）'}"))

    可疑 = _掃(BASE / "skills", 可疑樣式)
    結果.append(("❌" if 可疑 else "✅", "第三方 Skill 掃描（2-C 的三個習慣）",
               "；".join(可疑) if 可疑 else "簡易規則未命中；不代表 skills/ 中所有內容都安全"))

    sys.path.insert(0, str(BASE))
    from tools.run_code import 執行作業

    擋下 = 執行作業("import os\nos.listdir('.')").startswith("⛔")
    結果.append(("✅" if 擋下 else "❌", "run_code 有圍欄", "import os 會被擋下" if 擋下 else "import os 沒有被擋下！"))

    多agent = BASE / "deploy" / "multi_agent.py"
    內容 = 多agent.read_text(encoding="utf-8") if 多agent.exists() else ""
    結果.append(("✅" if "recursion_limit" in 內容 else "❌", "agent 有 recursion_limit", "每個批改子 agent 最多跑 25 步"))
    結果.append(("⚠️" if "SqliteSaver" in 內容 else "✅", "checkpointer 是持久的",
               "SQLite 只存在這一台機器；雲端機器會被換掉，上線要改用 PostgresSaver"))
    有確認 = "interrupt(" in 內容
    結果.append(("✅" if 有確認 else "❌", "不可逆的動作（寄信）前面有 interrupt", "寄信前會停下來等人確認" if 有確認 else "找不到 interrupt"))
    有驗證 = "稽核員人設" in 內容
    結果.append(("✅" if 有驗證 else "❌", "有獨立的驗證 agent", "批改和稽核的職責與輸入分開；獨立稽核仍可能漏錯" if 有驗證 else "找不到驗證 agent"))

    網址 = os.environ.get("HOMEWORK_MCP_URL")
    結果.append(("✅" if 網址 else "⚠️", "作業倉庫 MCP 從網址連（遠端 MCP）",
               f"HOMEWORK_MCP_URL={網址}" if 網址 else "現在是在同一台機器用 stdio 開 MCP server；跨機器使用時可設定遠端 MCP 網址，依服務配置授權"))

    缺 = [f for f in ("langgraph.json", "requirements.txt", "Dockerfile", ".dockerignore") if not (BASE / f).exists()]
    結果.append(("❌" if 缺 else "✅", "部署檔都在", f"缺少：{'、'.join(缺)}" if 缺 else "langgraph.json、requirements.txt、Dockerfile、.dockerignore"))
    return 結果


def 印出檢查結果() -> bool:
    結果 = 檢查()
    for 符號, 項目, 說明 in 結果:
        print(f"{符號} {項目}\n     {說明}")
    失敗 = sum(1 for r in 結果 if r[0] == "❌")
    print(f"\n{'🚫 有 ' + str(失敗) + ' 項沒過，先修好再上線' if 失敗 else '🎉 沒有 ❌，⚠️ 的項目上線前記得處理'}")
    return 失敗 == 0


if __name__ == "__main__":
    sys.exit(0 if 印出檢查結果() else 1)


Writing /content/review-bot/deploy/preflight.py


In [18]:
#@title ▶ 執行：上線前檢查
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
from deploy import preflight

通過 = preflight.印出檢查結果()
記下("2-E 練習 4：上線前檢查", [{"結果": a, "項目": b, "說明": c} for a, b, c in preflight.檢查()])

✅ 程式和手冊裡沒有寫死的金鑰
     金鑰應該放在 Secrets / Secret Manager，從環境變數讀
⚠️ 模型金鑰從環境變數來
     WORKSHOP_PROVIDER=（看 0. 環境設定 的選擇）；GOOGLE_API_KEY 沒設定（Colab 內建模型不需要，上雲端一定要）
✅ 第三方 Skill 掃描（2-C 的三個習慣）
     簡易規則未命中；不代表 skills/ 中所有內容都安全
✅ run_code 有圍欄
     import os 會被擋下
✅ agent 有 recursion_limit
     每個批改子 agent 最多跑 25 步
⚠️ checkpointer 是持久的
     SQLite 只存在這一台機器；雲端機器會被換掉，上線要改用 PostgresSaver
✅ 不可逆的動作（寄信）前面有 interrupt
     寄信前會停下來等人確認
✅ 有獨立的驗證 agent
     批改和稽核的職責與輸入分開；獨立稽核仍可能漏錯
⚠️ 作業倉庫 MCP 從網址連（遠端 MCP）
     現在是在同一台機器用 stdio 開 MCP server；跨機器使用時可設定遠端 MCP 網址，依服務配置授權
✅ 部署檔都在
     langgraph.json、requirements.txt、Dockerfile、.dockerignore

🎉 沒有 ❌，⚠️ 的項目上線前記得處理


### 👀 你看到了什麼

- ✅ 只代表通過這份範例的檢查，例如未命中列出的金鑰樣式、簡易掃描未發現所列可疑詞、`run_code` 有課堂範例的執行限制、agent 有 `recursion_limit`、
  寄信前有 `interrupt`、有獨立的驗證 agent、部署檔都在。
- ⚠️ 的是上雲端前還要處理的：
  - **金鑰**：這個 Gemini 部署範例需要有效的 API key，放進 Secret Manager（Colab 左邊 🔑「密鑰」就是它的縮小版）。
  - **checkpointer**：SQLite 只存在這一台機器，上線要換成 PostgresSaver。
  - **MCP**：現在是同一台機器用 stdio 開 server；跨機器使用時可改成遠端 MCP 的網址（`HOMEWORK_MCP_URL`），依服務設定授權；OAuth 是其中一種方式。
- 想再試一次 2-C：把 `detailed-feedback` 那本有問題的手冊放進 `skills/`，再跑一次檢查，看「第三方 Skill 掃描」會不會變成 ❌。

### 真的要部署的時候（課後參考，會產生費用，指令以各平台官方文件為準）

```bash
# 在 review-bot/ 資料夾裡
# 0. 把 .env.example 複製成 .env，填入自己的 GOOGLE_API_KEY（.env 不進 git）
# 1. 本機先跑通（需要 pip install "langgraph-cli[inmem]"）
langgraph dev

# 2a. 自己的容器：打包成映像，放到 Cloud Run 這類服務；金鑰用 Secret Manager 注入
docker build -t review-bot .
gcloud run deploy review-bot --source . --set-secrets=GOOGLE_API_KEY=review-bot-key:latest

# 2b. 代管：把這個資料夾推上 GitHub，在 LangSmith Deployment 選這個 repo 部署（它會讀 langgraph.json）
```

正式部署還需補上執行環境的管理：**每一步看得見（trace、eval、成本），權限受到適當限制（最小權限、沙箱、審 Skill、金鑰、人工確認）。**

## 收尾：存檔與交作業

- 💾 **存到雲端硬碟**：把 `/content/review-bot/`（你改過的手冊、程式、State 紀錄）存一份到自己的 Google 雲端硬碟。
- ⬇️ **下載 zip**：直接下載到電腦。
- 📸 **最後一格**：印出實驗格存下的結果。挑選相關紀錄截圖，再依作業要求補上觀察與理由。

這些是實驗格透過 `記下()` 保存的結果，並非每次執行的所有中間狀態。
同一執行環境的紀錄會累積，可能包含其他 notebook 的實驗；請按輸出標題辨認。

瀏覽器暫時斷線不一定會清掉檔案；執行階段被刪除或回收時，`/content/` 的檔案才會失去。
想保留手冊、程式和 checkpoint，請先備份。notebook 本身則用上方選單「檔案 → 在雲端硬碟中儲存副本」，兩種存檔用途不同。
回來操作時先確認執行階段仍在；若已重設，重跑「0. 環境設定」及本段準備格，再依需要還原自己備份的檔案。

In [19]:
#@title 💾 存到 Google 雲端硬碟（會跳出授權視窗）
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
存到雲端硬碟()

Mounted at /content/drive
✅ 已存到雲端硬碟：/content/drive/MyDrive/langgraph-skills-workshop/20261103-032826
   （notebook 本身請用上方選單「檔案 → 在雲端硬碟中儲存副本」）


In [20]:
#@title ⬇️ 下載 zip
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
下載壓縮檔()

✅ 已打包：/content/review-bot-20261103-032828.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
#@title 📸 印出已記錄的實驗結果（選取相關紀錄交作業）
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
印出歷程()

📸 共 7 筆紀錄（存在 /content/review-bot/history.jsonl）

===== 1. [03:26:08] 2-E 練習 1：stdio 接上作業倉庫 =====
  工具: ['list_homework', 'get_homework', 'send_email']
  學生: ['小明', '小華', '小美', '阿傑', '小芳']
  說明書token: 244

===== 2. [03:26:12] 2-E 練習 1：網址接上作業倉庫 =====
  網址: 'http://127.0.0.1:48879/mcp'
  工具: ['list_homework', 'get_homework', 'send_email']

===== 3. [03:26:38] 2-E 練習 2：整張分工圖（thread_id=期中-第一批） =====
  結果: [{'學生': '小美', '用到的工具': [], '驗證': {'通過': True, '原因': 'OK'}, '重寫次數': 0}]
  模型呼叫次數: 2
  token: 1034
  秒數: 25

===== 4. [03:26:44] 2-E 練習 2：決定「只寄驗證通過的」（thread_id=期中-第一批） =====
  寄件紀錄: ['（模擬）已寄給 xiaomei@example.com']

===== 5. [03:27:46] 2-E 練習 3：API 開始（thread_id=API-第一批） =====
  thread_id: 'API-第一批'
  待確認: {'要寄出的信': [{'學生': '小明', 'email': 'xiaoming@example.com', '驗證通過': True, '回饋': '語法：第 1 行少了右括號 )\n改進：先把這一行修好，再執行一次看看。'}, {'學生': '小華', 'email': 'xiaohua@example.com', '驗證通過': True, '回饋': '優點：函式結構清晰簡潔，參數與函式命名皆符合 Python 慣例，能順利完成兩數相加並輸出正確結果。\n\n改進：若想讓程式碼更完整且易於維護，可以參考以下調整：\n1. 加入型別提示（Type Hints）：建議改